# Automation and Affordability in U.S. Counties

Is the task composition of local work associated with what residents can afford?

Chris Bell  
Julian Pacheco

In [1]:
import os, sys
os.environ["R_HOME"]=os.path.join(sys.prefix, "lib", "R")
%load_ext rpy2.ipython
import rpy2.robjects as ro
from great_tables import style, loc
ro.r('''
suppressMessages({
  library(ggplot2)
  library(patchwork)
  library(scales)
})

# Semantic palette. Raw hex from Wes Anderson's Zissou1 (wesanderson package), adopted
# 13 Aug 2026 without running the dataviz skill's colorblind-separation/contrast validator —
# a colorblind-mode toggle is planned separately to cover that, per Julian's call. Names kept
# for role continuity; hues no longer match the names they're assigned to. GREY uses IsleofDogs1's
# neutral instead of Zissou1's own fifth color (a bright red, a bad fit for a "context" role named
# GREY) — that red is kept below as a reserve color instead of being discarded.
# BLUE substantive result, ORANGE failed specification,
# GREEN corrected or preferred specification, PINK flexible model, GREY context.
BLUE <- "#3B9AB2"
ORANGE <- "#78B7C5"
GREEN <- "#EBCC2A"
PINK <- "#E1AF00"
GREY <- "#8D8680"

# Reserve colors, not yet wired into any figure — pull from here if a new group needs its own
# color. All raw hex from various Wes Anderson palettes, same as the five above.
RED <- "#F21A00"       # Zissou1 (displaced from GREY)
PURPLE <- "#35274A"    # Rushmore1
FOREST <- "#0B775E"    # Rushmore1
BROWN <- "#79402E"     # IsleofDogs1
MAGENTA <- "#E6A0C4"   # GrandBudapest2

# Task group categorical palette. Two hue families (teal, orange) differentiated by
# lightness, replacing the four-distinct-hue scheme above for these four categorical
# series specifically. Scoped to task groups only — BLUE/ORANGE/GREEN/PINK/GREY above
# stay in place for unrelated semantic use elsewhere (specification comparisons,
# train/validation, etc). Purchasing power does not get its own swatch in this family;
# its continuous maps use viridis instead (see fig-afford-map), since packing a third
# teal shade in here failed the CVD/normal-vision separation check below.
# Validated 12 Aug 2026 with the dataviz skill's validate_palette.js, --pairs all mode
# (any two may sit adjacent, as in fig-taskarea's stacked bars): all checks pass, ΔE
# 17.4-19.8 on every pair under normal and CVD vision. The dark teal (RM_TEAL) had to
# shift toward blue to clear the chroma floor — a pure dark teal read as gray at low
# lightness in every candidate tried. Re-run the validator before changing these values:
# /Applications/quarto/bin/tools/aarch64/deno run --allow-all scripts/validate_palette.js
# "<hex,hex,hex,hex>" --mode light --pairs all   (from the dataviz skill's directory)
RC_TEAL <- "#1E9CAD"
RM_TEAL <- "#08599C"
NRC_ORANGE <- "#D9781F"
NRM_ORANGE <- "#8A3800"

# Task groups keep one color each wherever all four appear together.
TASK_COLORS <- c(
  "Non-Routine Cognitive" = NRC_ORANGE,
  "Non-Routine Manual" = NRM_ORANGE,
  "Routine Cognitive" = RC_TEAL,
  "Routine Manual" = RM_TEAL
)

dollar_axis <- function(v) {
  ifelse(is.na(v), "",
    ifelse(v == 0, "$0",
      sprintf("%s$%s", ifelse(v < 0, "-", ""),
              formatC(abs(v), format="d", big.mark=","))))
}

theme_set(
  theme_minimal(base_size=9.5) +
    theme(
      panel.background=element_rect(fill="#FCFCFB", color=NA),
      plot.background=element_rect(fill="#FCFCFB", color=NA),
      panel.grid.major=element_line(color="#E1E0D9", linewidth=0.35),
      panel.grid.minor=element_blank(),
      plot.title=element_text(face="bold", size=10.5, color="#0B0B0B", hjust=0.5, margin=margin(b=4)),
      plot.subtitle=element_text(size=8.5, color="#52514E", margin=margin(b=8)),
      strip.background=element_blank(),
      strip.text=element_text(face="bold", size=9, color="#0B0B0B"),
      axis.line=element_line(color="#C3C2B7", linewidth=0.3),
      axis.ticks=element_blank(),
      axis.text=element_text(size=8.5, color="#0B0B0B", face="bold"),
      axis.title=element_text(size=8.5, color="#0B0B0B"),
      legend.position="bottom",
      legend.title=element_blank(),
      legend.text=element_text(size=8.5, color="#0B0B0B"),
      legend.key=element_blank(),
      plot.margin=margin(10, 14, 8, 10)
    )
)
''')

def style_table(gt):
    """Shared table typography, applied to every GT table in the report so table
    text reads clearly smaller than body prose (great_tables defaults to 16px,
    matching body text)."""
    return gt.tab_options(
        table_font_size="12.5px",
        heading_title_font_size="14px",
        heading_subtitle_font_size="12px",
        column_labels_font_size="12.5px",
        source_notes_font_size="10.5px",
    ).tab_style(
        style=style.text(style="italic"),
        locations=loc.source_notes(),
    )

R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: package ‘ggplot2’ was built under R version 4.5.3 
  

# Introduction

American workers anticipate that automation will transform their jobs. According to Gallup, concern about technology rendering jobs obsolete rose between 2021 and 2023 ([Saad 2023](#ref-Gallup2023)), and Pew reports that Americans expect automation to displace other workers while leaving their own jobs largely intact ([Smith and Anderson 2017](#ref-Pew2017)). Concern of this kind is a belief about the future, and it need not track the work a place actually contains. Across the counties analyzed in this study, which together house approximately 282 million people, or about 84 percent of the United States population ([U.S. Census Bureau 2023b](#ref-census_popest2023)), routine work accounts for roughly 37 percent of measured task content, and that figure varies widely from county to county. This study asks whether a county’s task groups are associated with the purchasing power of the people who live there.

Three studies established the framework we build on. Autor, Levy, and Murnane ([2003](#ref-Autor2003)) showed that computers substitute for routine tasks and complement non-routine ones, which makes it possible to identify occupations exposed to automation. Applying that idea to local labor markets, Autor and Dorn ([2013](#ref-AutorDorn2013)) documented polarization in routine intensive areas, where employment grew at both the high and low ends of the wage distribution while middle wage employment declined. Acemoglu and Autor ([2011](#ref-AcemogluAutor2011)) then generalized the argument into a model in which tasks are the unit of production and skill groups compete to supply them.

None of this work speaks directly to purchasing power at the county level, for three reasons. Outcomes are reported in unadjusted wages and employment counts rather than in what those wages buy locally, the unit of analysis is the commuting zone rather than the county, and estimation is linear, so these designs cannot say whether purchasing power moves with task groups in fixed dollar steps or in proportion. None of the three is a flaw in the earlier studies, which were built for other questions. <a href="#sec-background" class="quarto-xref">Section 2</a> takes each one up in turn.

This study examines whether a county’s task groups are associated with its purchasing power, and whether any such association survives controls for poverty and unemployment. We build a county by year panel covering 2008 to 2023, excluding 2020, from datasets published by federal agencies. Each county year is described by four task groups, routine cognitive, routine manual, non-routine cognitive, and non-routine manual, following the task framework of Autor, Levy, and Murnane ([2003](#ref-Autor2003)) and the occupation taxonomy of Autor and Dorn ([2013](#ref-AutorDorn2013)). The outcome is purchasing power, measured in dollars as county median household income divided by the local price level from the Bureau of Economic Analysis Regional Price Parities. The panel holds 848 counties and 11,983 county year observations. <a href="#sec-data" class="quarto-xref">Section 3</a> describes the sources and how the panel was built.

We estimate a panel regression with year indicators and standard errors clustered by county, which recovers the association while accounting for the dependence among repeated observations of the same county. Residual plots, quantile quantile plots, and variance inflation factors test whether the specification holds. Those diagnostics show that purchasing power moves proportionally rather than in fixed dollar steps, and a log transformation captures most of that structure. Structured deviations remain in the residuals afterward, so we fit random forest and neural network models to test whether they find signal a linear form cannot. Complexity increases only where the diagnostics show a simpler specification falling short, and <a href="#sec-analysis" class="quarto-xref">Section 4</a> reports where each model landed. The central estimate is that a one percentage point shift toward non-routine manual work and away from non-routine cognitive work is associated with roughly \$846 less in purchasing power, with a 95 percent interval of plus or minus \$38, evaluated at the panel mean.

# Background

Automation does not affect every place in the same way because every place does not depend on the same kinds of work. Autor, Levy, and Murnane ([2003](#ref-Autor2003)) provided the framework for explaining why. Rather than treating occupations as either automated or not automated, they argued that technology replaces specific tasks within occupations. Routine tasks are easier to translate into explicit rules, while work that depends on judgment, adaptation, or direct interaction is harder to automate. This distinction produces the four task groups used in this study: routine cognitive, routine manual, non routine cognitive, and non routine manual.

Once work is understood through tasks, differences between local economies become measurable. Counties vary in the occupations their residents hold and therefore in the tasks their economies rely on. Autor and Dorn ([2013](#ref-AutorDorn2013)) showed that these differences persist across local labor markets and are associated with long term employment polarization. Areas with more routine work experienced declines in middle wage employment alongside growth at the lower and upper ends of the wage distribution. Their occupation taxonomy provides the basis for assigning occupations to the task groups used here.

While Autor and Dorn ([2013](#ref-AutorDorn2013)) showed how these task differences appear across local labor markets, Acemoglu and Autor ([2011](#ref-AcemogluAutor2011)) explained why those differences matter economically. In their model, tasks are the unit of production and workers with different skills compete to perform them. Technological change shifts which workers hold an advantage in particular tasks. This means that the economic effects of technology depend partly on the kinds of work a local economy relies on. This framework provides the theoretical basis for examining task groups at the county level.

A county’s task groups, however, are only one part of its economic conditions. Counties are also commonly described using measures such as poverty, unemployment, population, and household income. Each captures a different part of the local economy. Poverty identifies households with limited resources, unemployment reflects access to work, and household income describes the amount of money households receive. None of these measures alone provides a complete picture of the economic conditions residents face.

<a href="#fig-county-context" class="quarto-xref">Figure 1</a> provides an overview of this variation. Poverty (Panel A) and unemployment (Panel B) identify different concentrations of economic distress, while median household income (Panel C) highlights counties with especially high or low nominal resources. The patterns do not align perfectly across measures. A county that appears relatively strong under one indicator may appear less favorable under another. This variation is important because it shows that local economic conditions cannot be summarized by a single conventional measure.

In [2]:
import os
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine

engine = create_engine(
    os.environ["AUTORACK_URL"],
    pool_pre_ping=True,
    pool_recycle=300
)

In [3]:
context_year = 2023
context_df = pd.read_sql("""
    select cb.county_fips, cb.poverty_rate, cb.unemployment_rate,
           cb.median_household_income, ca.affordability_salary as purchasing_power
    from county_baseline cb
    join county_affordability ca
      on ca.county_fips = cb.county_fips and ca.year = cb.year
    where cb.year = %(yr)s
""", engine, params={"yr": context_year})
context_df["fips"] = context_df["county_fips"].astype(str).str.zfill(5)
# RPP recovered from the already computed purchasing power measure (@eq-afford, inverted):
# purchasing power = income / (RPP / 100), so RPP = income / purchasing power * 100.
context_df["rpp"] = context_df["median_household_income"] / context_df["purchasing_power"] * 100

gdf_context = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip"
)
gdf_context = gdf_context.rename(columns={"GEOID": "fips"})
merged_context = gdf_context.merge(
    context_df[["fips", "poverty_rate", "unemployment_rate",
                "median_household_income", "rpp"]],
    on="fips", how="left"
)
merged_context = merged_context[
    ~merged_context["STATEFP"].isin(["02", "15", "60", "66", "69", "72", "78"])
].copy()

pov_ctx_vmin = float(context_df["poverty_rate"].quantile(0.02))
pov_ctx_vmax = float(context_df["poverty_rate"].quantile(0.98))
unemp_ctx_vmin = float(context_df["unemployment_rate"].quantile(0.02))
unemp_ctx_vmax = float(context_df["unemployment_rate"].quantile(0.98))
inc_ctx_vmin = float(context_df["median_household_income"].quantile(0.02))
inc_ctx_vmax = float(context_df["median_household_income"].quantile(0.98))
rpp_ctx_vmin = float(context_df["rpp"].quantile(0.02))
rpp_ctx_vmax = float(context_df["rpp"].quantile(0.98))

os.makedirs("output", exist_ok=True)
merged_context[["fips", "poverty_rate", "unemployment_rate",
                "median_household_income", "rpp", "geometry"]].to_file(
    "output/county_context.geojson", driver="GeoJSON"
)

In [4]:
%%R -i pov_ctx_vmin -i pov_ctx_vmax -i unemp_ctx_vmin -i unemp_ctx_vmax -i inc_ctx_vmin -i inc_ctx_vmax -i rpp_ctx_vmin -i rpp_ctx_vmax -w 10 -h 9 -u in -r 150
suppressMessages(library(sf))

context_sf <- st_read("output/county_context.geojson", quiet=TRUE)

context_panel_theme <- theme_void() +
  theme(legend.position="bottom", legend.key.width=unit(1.1, "cm"),
        plot.title=element_text(face="bold", size=11, hjust=0.5),
        plot.background=element_rect(fill="#FCFCFB", color=NA))

bottom_guide <- guide_colorbar(direction="horizontal", title.position="bottom", title.hjust=0.5)

p_pov <- ggplot(context_sf) +
  geom_sf(aes(fill=poverty_rate), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=ORANGE, limits=c(pov_ctx_vmin, pov_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE", name="Poverty Rate (%)",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Poverty Rate")

p_unemp <- ggplot(context_sf) +
  geom_sf(aes(fill=unemployment_rate), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=PINK, limits=c(unemp_ctx_vmin, unemp_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE", name="Unemployment Rate (%)",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Unemployment Rate")

p_inc <- ggplot(context_sf) +
  geom_sf(aes(fill=median_household_income), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=GREY, limits=c(inc_ctx_vmin, inc_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE",
                      labels=function(v) sprintf("$%.0fk", v/1000),
                      name="Median Household Income",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Median Household Income")

p_rpp <- ggplot(context_sf) +
  geom_sf(aes(fill=rpp), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=BLUE, limits=c(rpp_ctx_vmin, rpp_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE",
                      name="Regional Price Parity",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Regional Price Parity")

(p_pov | p_unemp) / (p_inc | p_rpp) +
  plot_annotation(title="Four Views of County Economic Conditions",
                  tag_levels="A",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

The comparison becomes especially important when moving from income to purchasing power. Median household income measures how much money a household receives, but it does not account for what that money can buy locally. Counties with high nominal incomes may also face high housing and other costs. Lower income counties may have lower prices that allow the same amount of money to go further. Local prices can therefore change how the economic position of a county is interpreted.

Purchasing power brings these two pieces together by adjusting household income for differences in local prices. This adjustment does not simply raise or lower every county by the same amount. It can change how counties compare with one another. A county that appears especially prosperous based on income alone may move closer to the middle once its higher prices are considered, while another county may appear stronger after adjustment. For this study, purchasing power therefore provides a more informative measure of the economic resources available to residents than nominal income alone.

The geography used to measure these conditions also matters. Much of the previous literature examines commuting zones because they approximate local labor markets. That scale is well suited to studying employment adjustment, but it can combine counties with different local prices into a single labor market. Housing costs and other expenses can vary substantially between a metropolitan core and surrounding counties. A county level design preserves this variation and allows task groups to be compared with the prices residents face where they live.

A final consideration is how the relationship between task groups and purchasing power is represented. A relationship expressed only in fixed dollar changes assumes that the same shift in task groups corresponds to the same dollar difference everywhere. That may not match the way purchasing power varies across counties. If the relationship is proportional, the same change in task groups would correspond to a larger dollar difference where purchasing power is already high and a smaller difference where it is low. The analysis therefore evaluates the form of the relationship rather than assuming that a fixed dollar specification is appropriate from the outset.

Together, these considerations define the gap addressed in this study. Previous research establishes why task groups matter for local labor markets, while conventional measures such as poverty, unemployment, and household income describe different dimensions of county economic conditions. Adjusting income for local prices adds another dimension by showing what those resources can actually buy. The data and modeling strategy in <a href="#sec-data" class="quarto-xref">Section 3</a> bring these pieces together to examine whether differences in a county’s task groups are associated with differences in county purchasing power.

# Data

## Data collection

A longitudinal county analysis depends on measures that remain comparable across both geography and time. To meet that requirement, we combined five federal datasets published by three agencies, all retrieved in June 2026 through agency APIs or bulk file downloads and loaded into PostgreSQL. <a href="#tbl-sources" class="quarto-xref">Table 1</a> summarizes each source, the number of records retrieved, and the variables it contributes.

In [5]:
import pandas as pd
from great_tables import GT

sources_df=pd.DataFrame({
    "Source": ["Census SAIPE","BEA Regional Price Parities","Census CBSA delineation",
               "ACS 1 year estimates","BLS LAUS"],
    "Records retrieved": [50283, 884, 387, 15690, 88004],
    "Variables supplied": ["median household income, poverty rate",
                           "county price level relative to the national average",
                           "county to metropolitan area crosswalk",
                           "population, occupational employment by category",
                           "unemployment rate"],
    "Warehouse table": ["`county_baseline`","`cbsa_rpp`","`cbsa`",
                        "`county_baseline`, `county_task_exposure`","`county_baseline`"],
})

style_table(GT(sources_df)
  .tab_header(title="Federal Data Sources and Variables Supplied")
  .fmt_integer(columns="Records retrieved", use_seps=True)
  .fmt_markdown(columns="Warehouse table")
  .cols_align(align="right", columns="Records retrieved")
  .tab_source_note("All sources retrieved June 2026 through agency APIs or bulk file download."))

The five sources do not begin at the same geographic level. Most observations are reported by county and year, while Regional Price Parities are published at the metropolitan area level. To create a common county year structure, county Federal Information Processing Standards (FIPS) code and year are the primary join keys. Regional Price Parities require an additional step. Counties are first linked to their metropolitan area using the Census Core Based Statistical Area (CBSA) delineation file and then matched to the corresponding BEA price measure ([U.S. Bureau of Economic Analysis 2024](#ref-bea_rpp); [U.S. Census Bureau 2023a](#ref-census_cbsa)).

The study covers 2008 through 2023, excluding 2020. The American Community Survey (ACS) did not publish its standard 1 year estimates in 2020 following disruptions to data collection during the COVID pandemic ([U.S. Census Bureau 2024](#ref-census_acs)). Because these estimates provide the occupational employment counts used to construct the task groups, the same year is excluded from the panel. Across the remaining fifteen years, the Small Area Income and Poverty Estimates (SAIPE) program provides an initial frame of 47,140 county year observations.[1]

## Measuring task groups

Following the task framework introduced in <a href="#sec-background" class="quarto-xref">Section 2</a>, we grouped the broad occupational categories reported in the ACS 1 year estimates into four task groups: routine cognitive, routine manual, non routine cognitive, and non routine manual. Because the Census categories do not carry these labels directly, we applied the classification logic of Autor and Dorn ([2013](#ref-AutorDorn2013)). Clerical and sales occupations were classified as routine cognitive, production and construction occupations as routine manual, managerial, professional, and technical occupations as non routine cognitive, and service occupations as non routine manual. For each county and year, employment was then summed across the occupational categories assigned to each group, as shown in <a href="#eq-totals" class="quarto-xref">Equation 1</a>.

<span id="eq-totals">$$
T_{g,c,t} = \sum_{o \in g} E_{o,c,t}
 \qquad(1)$$</span>

Here $E_{o,c,t}$ represents employment in occupational category $o$ for county $c$ during year $t$, and the sum includes all occupations assigned to task group $g$. The calculation uses employment counts only and does not apply task intensity weights. As a result, each group measures the amount of county employment associated with that type of work. Dividing each group total by total employment, as shown in <a href="#eq-group" class="quarto-xref">Equation 2</a>, converts the four values to proportions between zero and one that sum to one. Together, these proportions describe how a county’s employment splits across the four task groups.

<span id="eq-group">$$
G_{g,c,t} = \frac{T_{g,c,t}}{\sum_{g'} T_{g',c,t}}
 \qquad(2)$$</span>

These proportions place counties of different sizes on a common scale, allowing the analysis to compare task groups rather than the size of the workforce. Because the four proportions sum to one, one group must be omitted from the regression to avoid perfect collinearity. Non routine cognitive work is the reference group, so the remaining coefficients are interpreted relative to it. <a href="#sec-analysis" class="quarto-xref">Section 4</a> explains how this composition is incorporated into the models.

We retain all four task groups in the data rather than combining them into a single routine intensity index because the type of work matters. Under a single index, a county shifting away from clerical work and a county shifting away from assembly work could appear identical even though the underlying changes in task groups are different.

A change in the source data also affects comparisons over time. The Census Bureau revised its occupational classification between 2009 and 2010, changing some category boundaries while preserving their assignment to the four task groups. <a href="#sec-analysis" class="quarto-xref">Section 4</a> examines whether this classification change affects the observed variation and reports the corresponding robustness checks.

## Measuring purchasing power

To account for differences in what household income can buy across counties, the outcome is measured as local purchasing power. We calculate purchasing power by adjusting median household income from Census SAIPE using Regional Price Parities from the Bureau of Economic Analysis. BEA reports these parities as an index with the national average set to 100. Dividing the index by 100 converts it to a local price multiplier, giving the purchasing power measure defined in <a href="#eq-afford" class="quarto-xref">Equation 3</a>.

<span id="eq-afford">$$
A_{c,t} = \frac{\text{Median household income}_{c,t}}{\text{RPP}_{c,t} / 100}
 \qquad(3)$$</span>

A county with a median household income of \$60,000 and a Regional Price Parity of 120 has purchasing power of \$50,000. In practical terms, that income buys roughly what \$50,000 would buy at national average prices. This adjustment makes household income more comparable across counties by accounting for differences in local prices.

Regional Price Parities adjust for differences across places, but not for changes in the national price level over time. Purchasing power is therefore reported in U.S. dollars. The regression includes year indicators to account for changes shared across counties within each year, including national price movement. <a href="#sec-analysis" class="quarto-xref">Section 4</a> explains how these year effects enter the model.

BEA does not publish a separate Regional Price Parity for counties outside metropolitan areas. These counties are assigned the corresponding state level parity instead. Within the analytical panel, 45.5 percent of county year observations use a metropolitan parity and 54.5 percent use the state measure. This provides price coverage across the full analytical sample, but it also introduces a limitation because counties assigned the same state value may face different local prices.

## Control variables

The models include three county level covariates to account for economic conditions that may be associated with both task groups and purchasing power. Poverty rate is drawn from Census SAIPE, unemployment rate from the Bureau of Labor Statistics (BLS) Local Area Unemployment Statistics (LAUS) program ([U.S. Bureau of Labor Statistics 2024](#ref-bls_laus)), and population from the ACS 1 year estimates. Poverty and unemployment capture differences in material hardship and labor market conditions across counties. Population accounts for variation in county scale and enters the models in logarithmic form because population size varies substantially across the analytical sample.

## The analytical dataset

The source frame contains 47,140 county year observations, but not all observations contain the occupational data required for estimation. Two restrictions define the final analytical sample.

The first is ACS occupational coverage. The Census Bureau publishes ACS 1 year occupational estimates only for areas with populations of at least 65,000. Counties below this threshold therefore lack the employment counts needed to construct the four task groups. Applying this restriction reduces the sample to 12,087 county year observations across 848 counties.

The second restriction requires complete values for the model covariates. An additional 104 observations are excluded because unemployment rates are missing, while population, median household income, and poverty rate are complete among the retained counties. The final analytical sample contains 11,983 county year observations across 848 counties. All models reported in <a href="#sec-analysis" class="quarto-xref">Section 4</a> and <a href="#sec-results" class="quarto-xref">Section 5</a> are estimated using these same observations.

For machine learning evaluation, the data are partitioned at the county level rather than by individual county year observations. Because each county can appear in the panel for up to fifteen years, a random row level split could place observations from the same county in both the training and test sets. Grouping the split by county prevents this overlap and ensures that all observations from a county remain entirely within one partition. <a href="#sec-analysis" class="quarto-xref">Section 4</a> describes the evaluation procedure in detail.

The population threshold also limits the scope of the results. The 848 retained counties contain about 84 percent of the United States population but represent a minority of all counties. They are generally more populous and metropolitan than the counties excluded by the ACS threshold, leaving rural counties underrepresented. The regression and machine learning results should therefore be interpreted as describing counties above the ACS publication threshold rather than counties nationwide. <a href="#sec-conclusions" class="quarto-xref">Section 6</a> returns to this limitation.

## Storage and organization

The data architecture separates raw source records from the tables used for analysis. Raw retrievals and intermediate results are stored in a PostgreSQL data lake containing 41 tables. These records are intentionally left unnormalized to preserve traceability and allow each processing step to be reproduced without returning to the original agency source.

Processed data are then written to an analytical warehouse containing eleven tables, with the core analytical tables organized in third normal form (3NF). Before entering the warehouse, county identifiers are standardized, records are restricted to the study period, jurisdictions outside the panel are removed, and foreign key constraints are enforced. This creates a consistent structure in which every analytical record can be linked to a valid county.

<a href="#fig-erd" class="quarto-xref">Figure 2</a> summarizes the full path from federal sources through the data lake and warehouse to the analysis ready tables. The figure emphasizes the eight warehouse tables used directly in the analysis and omits three supporting tables that are not required for the models reported here.

[1] The District of Columbia and Kalawao County, Hawaii, are excluded because they are absent from the county reference file. Connecticut counties leave the panel after 2021 because the state replaced its legacy counties with planning regions beginning in 2022.

In [6]:
import graphviz

def entity(name, rows):
    body = f'<TR><TD COLSPAN="3" BGCOLOR="#2A78D6"><FONT COLOR="white" POINT-SIZE="15"><B>{name}</B></FONT></TD></TR>'
    for typ, col, marker in rows:
        m = f'<FONT COLOR="#8F8D87" POINT-SIZE="10">{marker}</FONT>' if marker else ""
        body += f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="10">{typ}</FONT></TD><TD ALIGN="LEFT"><FONT POINT-SIZE="13">{col}</FONT></TD><TD ALIGN="LEFT">{m}</TD></TR>'
    return f'<<TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" CELLPADDING="3">{body}</TABLE>>'

CENSUS_FILL = "#DCEEF7"; CENSUS_BORDER = "#2A78D6"
BEA_FILL = "#DCF3EA"; BEA_BORDER = "#1BAF7A"
BLS_FILL = "#F7DCE6"; BLS_BORDER = "#C2255C"
OUT_FILL = "#FBE3D3"; OUT_BORDER = "#EB6834"
LAKE_FILL = "#EDEDEA"; LAKE_BORDER = "#8F8D87"

dot = f'''
digraph architecture {{
  rankdir=TB
  compound=true
  bgcolor="#FCFCFB"
  fontname="Helvetica"
  label=<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0"><TR><TD><FONT FACE="Helvetica-Bold" POINT-SIZE="24">Data Architecture, Source to Output</FONT></TD></TR></TABLE>>
  labelloc="t"
  nodesep=0.4
  ranksep=0.9
  node [fontname="Helvetica", shape=box, style="rounded", color="#C3C2B7"]
  edge [fontname="Helvetica", color="black", arrowsize=0.6]

  subgraph cluster_source {{
    label="DATA SOURCES"
    labeljust="l"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F9F9F7"
    margin=18
    node [style="filled,rounded", margin="0.1,0.05", fontsize=14]
    src_census [label="Census Bureau\\nSAIPE · ACS 1yr · CBSA delineation", fillcolor="{CENSUS_FILL}", color="{CENSUS_BORDER}"]
    src_bea [label="BEA\\nRegional Price Parities", fillcolor="{BEA_FILL}", color="{BEA_BORDER}"]
    src_bls [label="BLS\\nLAUS", fillcolor="{BLS_FILL}", color="{BLS_BORDER}"]
  }}

  subgraph cluster_lake {{
    label="DATA LAKE"
    labeljust="l"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F9F9F7"
    margin=18
    node [style="filled,rounded", fillcolor="{LAKE_FILL}", color="{LAKE_BORDER}", margin="0.1,0.05", fontsize=14]
    lake [label="Raw retrievals and intermediate results\\n41 tables, unnormalized"]
  }}

  subgraph cluster_warehouse {{
    label="DATA WAREHOUSE THIRD NORMAL FORM"
    labeljust="l"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F9F9F7"
    margin=18
    node [shape=plain]

    state [label={entity("state", [("varchar","state_code","PK"),("varchar","state_name","")])}]
    county [label={entity("county", [("bigint","county_fips","PK"),("varchar","county_name",""),("varchar","state_code","FK")])}]
    county_baseline [label={entity("county_baseline", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","population",""),("numeric","median_household_income",""),("numeric","poverty_rate",""),("numeric","unemployment_rate","")])}]
    county_task_exposure [label={entity("county_task_exposure", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","routine_cognitive",""),("numeric","routine_manual",""),("numeric","non_routine_cognitive",""),("numeric","non_routine_manual",""),("numeric","routine_cognitive_share","generated"),("numeric","routine_manual_share","generated"),("numeric","non_routine_cognitive_share","generated"),("numeric","non_routine_manual_share","generated")])}]
    county_affordability [label={entity("county_affordability", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","affordability_salary","")])}]
    county_cbsa_code [label={entity("county_cbsa_code", [("bigint","county_fips","PK,FK"),("varchar","cbsa_code","PK,FK")])}]
    cbsa [label={entity("cbsa", [("varchar","cbsa_code","PK"),("varchar","cbsa_name","")])}]
    cbsa_rpp [label={entity("cbsa_rpp", [("varchar","cbsa_code","PK,FK"),("int","year","PK"),("numeric","rpp_value","")])}]

    {{rank=same; county_baseline; county_task_exposure; county_affordability}}

    state -> county
    county -> county_baseline
    county -> county_task_exposure
    county -> county_affordability
    county -> county_cbsa_code
    county_cbsa_code -> cbsa
    cbsa -> cbsa_rpp
  }}

  subgraph cluster_output {{
    label="ANALYSIS OUTPUTS"
    labeljust="l"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F9F9F7"
    margin=18
    node [style="filled,rounded", fillcolor="{OUT_FILL}", color="{OUT_BORDER}", margin="0.1,0.05", fontsize=14]
    analysis_table [label="Analysis ready county-year table\\n(joined on county_fips and year)"]
    models [label="Statistical and ML models"]
    figures [label="Figures and tables"]
    {{rank=same; models; figures}}
    analysis_table -> models
    analysis_table -> figures
  }}

  src_census -> lake [ltail="cluster_source", lhead="cluster_lake"]
  src_bea -> lake [ltail="cluster_source", lhead="cluster_lake"]
  src_bls -> lake [ltail="cluster_source", lhead="cluster_lake"]
  lake -> state [ltail="cluster_lake", lhead="cluster_warehouse"]
  county_affordability -> analysis_table [ltail="cluster_warehouse", lhead="cluster_output"]
  county_baseline -> analysis_table [style=invis]
  county_task_exposure -> analysis_table [style=invis]
}}
'''
graphviz.Source(dot)

The warehouse is organized around the county table, which is keyed by FIPS code and linked to state as a reference table. Three analytical tables join to each county by FIPS code and year. county_baseline contains population, median household income, poverty rate, and unemployment rate. county_task_exposure contains the four task group totals defined in <a href="#eq-totals" class="quarto-xref">Equation 1</a>, while county_affordability contains the purchasing power outcome.

Two table names reflect earlier stages of the project. county_task_exposure was originally created for a measure based on Occupational Information Network (O\*NET) task ratings ([National Center for O\*NET Development, U.S. Department of Labor 2024](#ref-onet)), although the current table contains the employment based task measures used in this study. Likewise, county_affordability and its affordability_salary column predate the purchasing power terminology adopted in the report. The normalized task shares defined in <a href="#eq-group" class="quarto-xref">Equation 2</a> are stored as generated columns, allowing the database to derive them directly from the four task totals and maintain a consistent calculation across the analysis.

Regional Price Parities follow a separate relational path because BEA reports them at the metropolitan level. County membership in a metropolitan area is stored in the county_cbsa_code junction table, which links county to cbsa and then to cbsa_rpp. Keeping these relationships separate preserves the geographic level at which BEA reports the price measures and avoids duplicating the same metropolitan parity across multiple county records.

Together, these steps produce a consistent county year panel that integrates a county’s task groups, purchasing power, and baseline economic conditions within a common analytical structure. This structure allows the analysis to compare how task groups and purchasing power vary across counties and over time.

# Analysis

In [7]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

engine=create_engine(os.environ["AUTORACK_URL"], pool_pre_ping=True, pool_recycle=300)
df=pd.read_sql("""
select ca.county_fips, ca.year, ca.affordability_salary,
    cte.routine_cognitive_share, cte.routine_manual_share,
    cte.non_routine_cognitive_share, cte.non_routine_manual_share,
    cb.poverty_rate, cb.unemployment_rate, cb.population
from county_affordability ca
join county_task_exposure cte on ca.county_fips=cte.county_fips and ca.year=cte.year
join county_baseline cb on ca.county_fips=cb.county_fips and ca.year=cb.year
""", engine)
df["log_population"]=np.log(df["population"])

In [8]:
# level model: OLS with year indicators, standard errors clustered by county
reg_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share",
          "poverty_rate","unemployment_rate","log_population"]

reg=df.dropna(subset=reg_cols+["affordability_salary"]).copy()

X_panel=pd.concat([reg[reg_cols],
                   pd.get_dummies(reg["year"], prefix="year", drop_first=True).astype(float)], axis=1)
X_panel=sm.add_constant(X_panel)

model=sm.OLS(reg["affordability_salary"], X_panel).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})

reg["fitted"]=model.fittedvalues
reg["resid"]=model.resid

# log respecification, same design matrix
reg["actual_log"]=np.log(reg["affordability_salary"])
model_log=sm.OLS(reg["actual_log"], X_panel).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})
reg["resid_log"]=model_log.resid
reg["fitted_log"]=model_log.fittedvalues

## The unadjusted relationship

The ladder starts with the relationship itself, before any controls. <a href="#fig-univariate" class="quarto-xref">Figure 3</a> plots purchasing power against each of the four task groups across the panel. The two manual groups slope down most steeply, non-routine cognitive slopes up, and routine cognitive is the flattest of the four. None of the four is tight enough to carry a claim on its own, which is the point of the models that follow, but the ordering visible here is the ordering the adjusted estimates preserve.

In [9]:
uni_labels={
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_cognitive_share":"Non-Routine Cognitive",
    "non_routine_manual_share":"Non-Routine Manual",
}
uni_pts=df.melt(id_vars="affordability_salary", value_vars=list(uni_labels.keys()),
                var_name="group", value_name="share").dropna()
uni_pts["group"]=uni_pts["group"].map(uni_labels)
uni_pts["share"]=uni_pts["share"]*100  # express as percent of county employment

uni_pts["bin"]=uni_pts.groupby("group")["share"].transform(
    lambda s: pd.qcut(s, 20, labels=False, duplicates="drop"))
uni_bins=(uni_pts.groupby(["group","bin"])[["share","affordability_salary"]]
          .mean().reset_index())
uni_order=list(uni_labels.values())

In [10]:
%%R -i uni_pts -i uni_bins -i uni_order -w 10 -h 6 -u in -r 150
uni_pts$group <- factor(uni_pts$group, levels=unlist(uni_order))
uni_bins$group <- factor(uni_bins$group, levels=unlist(uni_order))

# one color per task group, matching the palette used everywhere else in the report
GROUP_COLORS <- TASK_COLORS

ggplot(uni_pts, aes(x=share, y=affordability_salary)) +
  geom_point(alpha=0.08, size=0.25, color="grey40") +
  geom_line(data=uni_bins, aes(color=group), linewidth=0.8, show.legend=FALSE) +
  geom_point(data=uni_bins, aes(color=group), size=1.8, show.legend=FALSE) +
  facet_wrap(~group, scales="free_x", ncol=2) +
  scale_color_manual(values=GROUP_COLORS) +
  scale_x_continuous(breaks=scales::pretty_breaks(n=8)) +
  scale_y_continuous(labels=function(v) sprintf("$%s", formatC(v, format="d", big.mark=","))) +
  labs(title="Purchasing Power and Task Groups, Unadjusted Relationship",
       x="Percent of County Employment", y="Purchasing Power") +
  theme(plot.title=element_text(hjust=0.5, size=15),
        strip.text=element_text(size=12),
        axis.title=element_text(size=12),
        axis.text=element_text(size=10.5),
        axis.ticks.x=element_line(color="#C3C2B7", linewidth=0.3),
        panel.spacing=unit(1.4, "lines"))

## Model specification

The four task group values defined in <a href="#eq-group" class="quarto-xref">Equation 2</a> sum to one, so all four cannot enter a regression together. One serves as the reference group, and we use non-routine cognitive. Each remaining coefficient is the change in purchasing power associated with a shift out of non-routine cognitive work and into that group, holding the controls fixed, and each is therefore a comparison against the reference rather than a standalone quantity. The group values run on a zero to one scale, so the per percentage point figures reported later divide the estimated coefficients by 100.

Year indicators capture conditions common to all counties within a year. These include the 2008 recession, the recovery through the 2010s, and the price movement that purchasing power does not itself account for. Because purchasing power is not deflated to a constant base year, the year effects carry price level drift alongside any other national change, and we treat them as nuisance parameters rather than as evidence of rising purchasing power. The task group coefficients compare counties within a year and are unaffected by this.

Standard errors are clustered by county, because repeated observations of the same county across fifteen years are not independent draws. A Durbin Watson statistic of 0.496 is consistent with that dependence. <a href="#sec-results" class="quarto-xref">Section 5</a> reports the estimated coefficients.

## Collinearity and the reference group

The unnormalized employment totals of <a href="#eq-totals" class="quarto-xref">Equation 1</a> cannot support a regression. A variance inflation factor measures how much a coefficient is destabilized by its correlation with the other inputs, and a value above 10 is the conventional threshold for concern. The four totals carry factors between 13 and 27, because all four are employment counts that scale with county size, so they move together and carry little independent information. The specification estimated fixes this twice over, as <a href="#fig-vif" class="quarto-xref">Figure 4</a> shows. Normalizing by the four group total, as <a href="#sec-data" class="quarto-xref">Section 3</a> describes, removes the shared scale, and dropping non-routine cognitive as the reference group removes the constraint that the four proportions sum to one. Every factor on the estimated specification sits between 1.2 and 1.6.

In [11]:
raw=pd.read_sql("""
    select routine_cognitive, routine_manual, non_routine_cognitive, non_routine_manual
    from county_task_exposure
    """, engine)

raw_cols=["routine_cognitive","routine_manual","non_routine_cognitive","non_routine_manual"]
X_raw=sm.add_constant(raw[raw_cols])
vif_raw=pd.Series([variance_inflation_factor(X_raw.values, i) for i in range(1, X_raw.shape[1])],
                  index=raw_cols)

X_vif_level=sm.add_constant(reg[reg_cols])
vif_share=pd.Series([variance_inflation_factor(X_vif_level.values, i) for i in range(1, X_vif_level.shape[1])],
                    index=reg_cols)

group_display=["Routine Cognitive","Routine Manual","Non-Routine Cognitive","Non-Routine Manual"]

vif_dot_df=pd.concat([
    pd.DataFrame({"group":group_display, "vif":vif_raw.values,
                  "spec":"Unnormalized Employment Totals"}),
    pd.DataFrame({"group":["Routine Cognitive","Routine Manual","Non-Routine Manual"],
                  "vif":vif_share[["routine_cognitive_share","routine_manual_share",
                                   "non_routine_manual_share"]].values,
                  "spec":"Group Proportions, as Estimated"}),
], ignore_index=True)

# wide form for the before-to-after arrow on each row (excludes the reference group,
# which has no estimated point)
vif_seg_df=pd.DataFrame({
    "group":["Routine Cognitive","Routine Manual","Non-Routine Manual"],
    "raw":vif_raw[["routine_cognitive","routine_manual","non_routine_manual"]].values,
    "share":vif_share[["routine_cognitive_share","routine_manual_share",
                       "non_routine_manual_share"]].values,
})

In [12]:
%%R -i vif_dot_df -w 8 -h 4 -u in -r 150
vif_dot_df$group <- factor(vif_dot_df$group,
    levels=rev(c("Routine Cognitive","Routine Manual",
                 "Non-Routine Cognitive","Non-Routine Manual")))
vif_dot_df$spec <- factor(vif_dot_df$spec,
    levels=c("Group Proportions, as Estimated","Unnormalized Employment Totals"),
    labels=c("Group proportions","Raw totals"))

ref_label_df <- data.frame(group=factor("Non-Routine Cognitive", levels=levels(vif_dot_df$group)),
                            x=1, label="Reference category")

ggplot(vif_dot_df, aes(x=vif, y=group)) +
  geom_vline(xintercept=5, linetype="dotted", linewidth=0.35, color="#0B0B0B") +
  geom_vline(xintercept=10, linetype="dashed", linewidth=0.7, color="#0B0B0B") +
  annotate("text", x=5, y=Inf, label="VIF = 5", hjust=-0.15, vjust=1.4, color="grey45", size=3.1) +
  annotate("text", x=10, y=Inf, label="VIF = 10", hjust=-0.15, vjust=1.4, color="grey30", size=3.3, fontface="bold") +
  geom_line(aes(group=group), color="#0B0B0B", linewidth=0.6) +
  geom_point(aes(color=spec), size=3.2) +
  geom_text(aes(label=sprintf("%.1f", vif), color=spec), vjust=-1.1, size=3.2, show.legend=FALSE) +
  geom_text(data=ref_label_df, aes(x=x, y=group, label=label), inherit.aes=FALSE,
            hjust=0, vjust=-1.1, size=3.2, color="grey40", fontface="italic") +
  scale_x_continuous(limits=c(0, 30), breaks=seq(0, 30, by=5)) +
  scale_color_manual(values=c("Group proportions"=GREEN, "Raw totals"=ORANGE)) +
  labs(title="Variance Inflation Factors by Task Specification",
       x="Variance Inflation Factor", y=NULL, color=NULL) +
  theme(legend.position="bottom",
        plot.title=element_text(size=15),
        axis.ticks.x=element_line(color="#0B0B0B", linewidth=0.6),
        axis.ticks.length.x=unit(6, "pt"))

This also explains a problem visible earlier in the project. Before normalization the coefficients were unstable and their standard errors were large relative to their magnitudes, both standard symptoms of multicollinearity.

## Distribution of the analytical variables

With the specification set, we turn to the shape of the variables that enter it. <a href="#fig-dists" class="quarto-xref">Figure 5</a> shows the distributions of purchasing power and the four task groups across the task group panel, looking for the skewness and outliers that would shape the diagnostics below. The variance inflation factors above already establish that no two groups carry the same information once the proportions are used.

In [13]:
dist_label_map={
    "affordability_salary":"Purchasing Power ($)",
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_cognitive_share":"Non-Routine Cognitive",
    "non_routine_manual_share":"Non-Routine Manual",
}
dists_df=df.melt(value_vars=list(dist_label_map.keys()),
                 var_name="variable", value_name="value")
dists_df["variable"]=dists_df["variable"].map(dist_label_map)

In [14]:
%%R -i dists_df -w 10 -h 5.5 -u in -r 150
pp_vals <- subset(dists_df, variable=="Purchasing Power ($)")$value
pp_ylim <- c(0, max(pp_vals) * 1.02)

p_pp <- ggplot(subset(dists_df, variable=="Purchasing Power ($)"), aes(x=variable, y=value)) +
  geom_violin(fill=BLUE, color=BLUE, alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", color="grey30", alpha=0.7,
               outlier.size=0.8, outlier.alpha=0.5) +
  scale_y_continuous(labels=dollar_axis) +
  coord_cartesian(ylim=pp_ylim) +
  labs(title="Purchasing Power ($)", x=NULL, y="Purchasing Power") +
  theme(axis.text.x=element_blank(), axis.ticks.x=element_blank(),
        plot.margin=margin(3, 6, 3, 6))

task_box_df <- subset(dists_df, variable %in% c("Routine Cognitive", "Routine Manual",
                                                 "Non-Routine Cognitive", "Non-Routine Manual"))
task_box_df$variable <- factor(task_box_df$variable,
    levels=c("Non-Routine Cognitive", "Non-Routine Manual", "Routine Cognitive", "Routine Manual"))

p_tasks <- ggplot(task_box_df, aes(x=variable, y=value, fill=variable, color=variable)) +
  geom_violin(alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", alpha=0.7,
               outlier.size=0.6, outlier.alpha=0.4) +
  scale_fill_manual(values=TASK_COLORS, guide="none") +
  scale_color_manual(values=TASK_COLORS, guide="none") +
  scale_y_continuous(labels=percent) +
  coord_cartesian(ylim=c(0.05, 0.65)) +
  scale_x_discrete(labels=function(x) gsub(" ", "\n", x)) +
  labs(title="Task Groups", x=NULL, y=NULL) +
  theme(axis.text.x=element_text(size=8))

(p_pp | p_tasks) + plot_layout(widths=c(1, 1.6)) +
  plot_annotation(title="Distribution of Purchasing Power and Task Groups",
                  tag_levels="A",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

Purchasing power is right skewed, with a skewness of 1.17. The center of the distribution sits near \$57,000, and a long tail of high purchasing power counties extends above it. That skew is why the diagnostics below concentrate on the extremes.

## Specification diagnostics

A linear model assumes a constant rate of association across the data range and errors that are symmetric and evenly spread around the fit. We test those assumptions with three diagnostics, the residuals plotted against fitted values, a QQ plot of the residuals, and the variance inflation factors already reported. The raw scale model shows clear systematic curvature in the residuals and a substantial upper tail departure in the QQ plot, indicating that the linear specification is poorly suited to the untransformed outcome (<a href="#fig-diagnostics" class="quarto-xref">Figure 7</a>, Panels A and B), alongside the same diagnostics after the log respecification below (Panels C and D). The model underestimates both the lowest and the highest purchasing power counties while fitting the middle well, and the upper tail departure reflects a residual skew of 1.01.

The level model still supplies the interpretable dollar estimates we report, and its own diagnostics show that the data hold structure a straight line in dollars cannot represent. These diagnostics are what drive the rest of the section. Curvature in the residual smoother and widening spread with fitted values are evidence that the fixed slope specification is wrong in a specific way, not merely imperfect, and each subsequent model is an attempt to relax the assumption the diagnostics identify. Taken together, the diagnostics support one conclusion: a fixed dollar relationship does not adequately describe the data.

## Log respecification

The level model’s failure suggests that a constant dollar association is a poor description of the relationship. Logging the outcome tests the alternative, that the associations are proportional, so the same shift in an input moves a high purchasing power county by more dollars than a low one. If that description fits better, the transformation should resolve the residual pattern the level model leaves behind. <a href="#fig-log-transform" class="quarto-xref">Figure 6</a> shows the outcome before (Panel A) and after (Panel B) the transformation.

In [15]:
level_label=f"Level, Skew {reg['affordability_salary'].skew():.2f}"
log_label=f"Logged, Skew {reg['actual_log'].skew():.2f}"
logt_df=pd.concat([
    pd.DataFrame({"value":reg["affordability_salary"], "dist":level_label}),
    pd.DataFrame({"value":reg["actual_log"], "dist":log_label}),
], ignore_index=True)
logt_order=[level_label, log_label]

In [16]:
%%R -i logt_df -i logt_order -w 11 -h 3.5 -u in -r 150
logt_df$dist <- factor(logt_df$dist, levels=unlist(logt_order))
level_label <- unlist(logt_order)[1]
log_label   <- unlist(logt_order)[2]

p_level <- ggplot(subset(logt_df, dist==level_label), aes(x=value)) +
  geom_histogram(bins=50, fill=ORANGE) +
  scale_x_continuous(labels=dollar_axis) +
  labs(title=level_label, x=NULL, y="Count")

p_log <- ggplot(subset(logt_df, dist==log_label), aes(x=value)) +
  geom_histogram(bins=50, fill=GREEN) +
  labs(title=log_label, x=NULL, y="Count")

(p_level | p_log) +
  plot_annotation(title="Purchasing Power Before and After the Log Transform",
                  tag_levels="A",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

Refitting the same specification on log purchasing power resolves most of the failure. <a href="#fig-diagnostics" class="quarto-xref">Figure 7</a> compares the residual and QQ diagnostics for both specifications directly. After logging purchasing power (Panels C and D), the residual pattern becomes substantially flatter and the QQ points track the reference line more closely, though some tail deviation remains; the residual skew falls from 1.01 to 0.09. The log linear fit is the baseline against which we evaluate the flexible models below.

In [17]:
resid_df=reg[["fitted","resid"]].copy()
resid_log_df=reg[["fitted_log","resid_log"]].copy()

In [18]:
%%R -i resid_df -i resid_log_df -w 9 -h 7 -u in -r 150
resid_df$std_resid <- as.numeric(scale(resid_df$resid))
resid_log_df$std_resid <- as.numeric(scale(resid_log_df$resid_log))

p1 <- ggplot(resid_df, aes(x=fitted, y=resid)) +
  geom_bin2d(bins=55, aes(fill=after_stat(count))) +
  scale_fill_gradient(low="#E8EEF4", high=BLUE, guide="none") +
  geom_hline(yintercept=0, linewidth=0.4, linetype="dashed", color="grey40") +
  geom_smooth(method="loess", formula=y~x, se=FALSE, color=ORANGE, linewidth=0.9) +
  scale_x_continuous(labels=dollar_axis) +
  scale_y_continuous(labels=dollar_axis) +
  labs(x=NULL, y="Residual, Raw", title="Residuals vs. Fitted")

p2 <- ggplot(resid_df, aes(sample=std_resid)) +
  stat_qq(color=BLUE, alpha=0.35, size=0.8) +
  stat_qq_line(color=ORANGE, linewidth=0.8) +
  labs(x=NULL, y="Standardized Residual", title="Normal Q-Q")

p3 <- ggplot(resid_log_df, aes(x=fitted_log, y=resid_log)) +
  geom_bin2d(bins=55, aes(fill=after_stat(count))) +
  scale_fill_gradient(low="#E8F4EE", high=GREEN, guide="none") +
  geom_hline(yintercept=0, linewidth=0.4, linetype="dashed", color="grey40") +
  geom_smooth(method="loess", formula=y~x, se=FALSE, color=PINK, linewidth=0.9) +
  labs(x="Fitted Purchasing Power", y="Residual, Log")

p4 <- ggplot(resid_log_df, aes(sample=std_resid)) +
  stat_qq(color=GREEN, alpha=0.35, size=0.8) +
  stat_qq_line(color=PINK, linewidth=0.8) +
  labs(x="Theoretical Quantiles", y="Standardized Residual")

(p1 | p2) / (p3 | p4) +
  plot_annotation(title="Regression Diagnostics, Raw vs. Log Specification",
                  tag_levels="A",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

<a href="#fig-binned-scale" class="quarto-xref">Figure 8</a> puts the same comparison in the units of the outcome. Fitted values are cut into twenty equal count bins, and each point compares a bin’s mean prediction with its mean actual value, so points on the line indicate no systematic bias in that range. On the level scale the model tracks purchasing power closely through the middle of the distribution, where most counties sit, while both tail bins come in roughly \$8,000 to \$9,000 above their predictions. On the log scale the points track the line throughout.

In [19]:
reg.loc[:, "bin"]=pd.qcut(reg["fitted"], 20, labels=False)
binned=reg.groupby("bin")[["fitted","affordability_salary"]].mean()
reg.loc[:, "bin_log"]=pd.qcut(reg["fitted_log"], 20, labels=False)
binned_log=reg.groupby("bin_log")[["fitted_log","actual_log"]].mean()

pts_df=pd.concat([
    pd.DataFrame({"fitted":reg["fitted"], "actual":reg["affordability_salary"],
                  "target":"Level Target"}),
    pd.DataFrame({"fitted":reg["fitted_log"], "actual":reg["actual_log"],
                  "target":"Log Target"}),
], ignore_index=True)
bins_df=pd.concat([
    pd.DataFrame({"fitted":binned["fitted"], "actual":binned["affordability_salary"],
                  "target":"Level Target"}),
    pd.DataFrame({"fitted":binned_log["fitted_log"], "actual":binned_log["actual_log"],
                  "target":"Log Target"}),
], ignore_index=True)

In [20]:
%%R -i pts_df -i bins_df -w 13 -h 5.5 -u in -r 150
pts_df$target <- factor(pts_df$target, levels=c("Level Target","Log Target"))
bins_df$target <- factor(bins_df$target, levels=c("Level Target","Log Target"))

ggplot(pts_df, aes(x=fitted, y=actual)) +
  geom_point(alpha=0.05, size=0.3, color="grey50") +
  geom_abline(slope=1, intercept=0, linewidth=0.4) +
  geom_point(data=bins_df, aes(color=target), size=2.4, show.legend=FALSE) +
  facet_wrap(~target, scales="free") +
  scale_color_manual(values=c("Level Target"=ORANGE,"Log Target"=GREEN)) +
  labs(title="Binned Fit Accuracy, Raw vs. Log Specification",
       x="Fitted Purchasing Power", y="Actual Purchasing Power")

The smoothed residual line does not lie perfectly flat even after the transformation. What remains could be nonlinearity, interactions among the inputs, or a variable the model does not contain, and the residuals alone cannot separate the three. That is exactly the situation a flexible model is built for. A random forest searches all three at once without being told in advance which to look for, so it tests the possibilities the diagnostics raise but cannot settle. It also fixes the standard the forest must meet, which is finding signal in held out counties that the log linear fit does not already capture.

## Variance decomposition

Purchasing power and the task groups vary both across counties and within a county over time, and the two carry different implications. <a href="#fig-between-within" class="quarto-xref">Figure 9</a> splits each group’s variation into the two components. The manual groups vary almost entirely between counties, which is what makes the purchasing power differences they carry durable rather than temporary. Routine cognitive is the only group with substantial within county movement.

In [21]:
df10=df[df["year"]>=2010].copy()
input_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share"]

within_vals=[]
for col in input_cols:
    yd=df10[col]-df10.groupby("year")[col].transform("mean")
    within=(yd-yd.groupby(df10["county_fips"]).transform("mean")).var()
    within_vals.append(within/yd.var()*100)

bw_labels=["Routine Cognitive","Routine Manual","Non-Routine Manual"]
bw_df=pd.concat([
    pd.DataFrame({"group":bw_labels, "value":[100-v for v in within_vals],
                  "component":"Between Counties"}),
    pd.DataFrame({"group":bw_labels, "value":within_vals,
                  "component":"Within Counties Over Time"}),
], ignore_index=True)
bw_text_df=pd.concat([
    pd.DataFrame({"group":bw_labels, "x":[(100-v)/2 for v in within_vals],
                  "label":[f"{100-v:.0f}%" for v in within_vals], "component":"Between Counties"}),
    pd.DataFrame({"group":bw_labels, "x":[100-v/2 for v in within_vals],
                  "label":[f"{v:.0f}%" for v in within_vals], "component":"Within Counties Over Time"}),
], ignore_index=True)
bw_annot_df=pd.DataFrame({
    "x":[(100-within_vals[0])/2, 100-within_vals[0]/2],
    "label":["Between Counties","Within Counties Over Time"],
})

In [22]:
%%R -i bw_df -i bw_text_df -i bw_annot_df -w 9 -h 3.5 -u in -r 150
bw_df$group <- factor(bw_df$group,
    levels=rev(c("Routine Cognitive","Routine Manual","Non-Routine Manual")))
bw_df$component <- factor(bw_df$component,
    levels=c("Within Counties Over Time","Between Counties"))
bw_text_df$group <- factor(bw_text_df$group, levels=levels(bw_df$group))

# Routine Cognitive's full color is a light teal, so white text loses contrast there
# even on the saturated "within counties" segment; every other segment stays dark-on-light
# or light-on-dark as before.
bw_text_df$text_color <- ifelse(bw_text_df$component=="Within Counties Over Time" &
                                   bw_text_df$group != "Routine Cognitive",
                                 "white", "#0B0B0B")

ggplot(bw_df, aes(x=value, y=group, fill=group, alpha=component)) +
  geom_col(width=0.7) +
  geom_text(data=bw_text_df, aes(x=x, y=group, label=label, color=text_color),
            inherit.aes=FALSE, size=3.4) +
  geom_text(data=bw_annot_df, aes(x=x, y=3.68, label=label),
            inherit.aes=FALSE, size=3.2, fontface="bold", color="#52514E") +
  scale_color_identity() +
  scale_fill_manual(values=TASK_COLORS, guide="none") +
  scale_alpha_manual(values=c("Between Counties"=0.32,
                              "Within Counties Over Time"=1),
                     guide="none") +
  scale_y_discrete(expand=expansion(add=c(0.6, 1.0))) +
  labs(title="Between-County vs. Within-County Variation by Task Group",
       x="Share of Variance (%)", y=NULL)

The Census occupation coding change between 2009 and 2010, described in <a href="#sec-data" class="quarto-xref">Section 3</a>, inflates apparent within county variation for the routine cognitive group. That inflation comes from how the source data were coded rather than from any property of the counties, so the figure is computed on the panel restricted to 2010 onward with each year’s cross county mean removed. We also refit the main estimates on pre pandemic (2010 to 2019) and post pandemic (2021 to 2023) windows, and <a href="#sec-results" class="quarto-xref">Section 5</a> reports coefficient stability across them.

## Predictive modeling

Model complexity follows a ladder, from linear models through tree ensembles to neural networks. The rule is to start with the simplest model that could plausibly work, move up a rung only while held out error improves, and stop when training and validation performance agree. The level model’s failed diagnostics justified the first step, the log respecification. The structured residuals that remain justify testing the next rung, a random forest, which represents interactions and curvature without requiring either to be specified in advance. A small neural network sits one rung above the forest and bounds the search, since if the forest finds no additional signal, the network confirms whether that ceiling is real. Every rung is scored on the same held out counties from the grouped split, so a gain at any rung is attributable to the model rather than to the rows it saw. Under this rule, finding no improvement is itself a result, because it establishes that the association is close to proportional and that the log linear model is the right stopping point.

We reuse the county year panel assembled above, extended with state identifiers for the comparison below.

In [23]:
# state_code isn't in the panel regression's df, add it here
state_lookup=pd.read_sql(
    "select c.county_fips, c.state_code, s.state_name "
    "from county c join state s on c.state_code=s.state_code", engine
)
state_name_map=state_lookup.drop_duplicates("state_code").set_index("state_code")["state_name"]
df_ml=df.merge(state_lookup, on="county_fips", how="left")

### Feature set and reference groups

In [24]:
feature_cols=[
    "routine_cognitive_share",
    "routine_manual_share",
    "non_routine_cognitive_share",
    "non_routine_manual_share",
    "poverty_rate",
    "unemployment_rate",
    "year",
    "state_code",
]

model_df=df_ml.dropna(subset=feature_cols).reset_index(drop=True).copy()

We define each task group as its proportion of employment across the four categories, so the four values already sum to one. That constraint creates perfect collinearity among the four measures, so one category must be dropped as the reference group. As with the panel regression, we retain `non_routine_cognitive_share` as the omitted reference, so coefficients on the remaining task groups are interpreted as shifts toward each group and away from non-routine cognitive work.

### Grouped train and test split

This is not a forecasting exercise, so a time ordered split is not required for its usual reason. A grouped split is still necessary, because a county’s economic profile barely moves year to year, which makes its 2018 and 2019 rows near duplicates. If two adjacent rows land on opposite sides of the split, the model can recall a county rather than learn a relationship. We therefore group by `county_fips` and split on row membership before any further feature engineering, so every test set prediction comes from a county the model has seen no rows of, which tests whether the pattern generalizes rather than whether it was memorized. The columns that make up the feature matrix are decided next and applied to this same split.

In [25]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

target_col="affordability_salary"
groups=model_df["county_fips"]

gss=GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx=next(gss.split(model_df, groups=groups))

groups_train=groups.iloc[train_idx]

assert len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))==0

### Year and state indicators

Treating year as a factor rather than as a numeric variable gives each year its own coefficient, with no assumption about ordering or spacing between them. One state serves as a reference in the same way. Every state with fewer than five distinct counties in the panel is relabeled as OTHER, since those states cannot support a stable coefficient of their own.

In [26]:
model_df["year"]=model_df["year"].astype("category")
model_df["state_code"]=model_df["state_code"].astype("category")

REFERENCE_YEAR=model_df["year"].cat.categories.min()  # earliest year as reference

# pool states with too few counties to support a stable, independent coefficient
county_counts=model_df.groupby("state_code", observed=True)["county_fips"].nunique()
THIN_STATE_THRESHOLD=5
thin_states=county_counts[county_counts<THIN_STATE_THRESHOLD].index.tolist()

model_df["state_code_grouped"]=model_df["state_code"].astype(str)
model_df["state_code_grouped"]=model_df["state_code_grouped"].where(
    ~model_df["state_code_grouped"].isin(thin_states), "OTHER"
)
model_df["state_code_grouped"]=model_df["state_code_grouped"].astype("category")

state_affordability=model_df.groupby("state_code_grouped", observed=True)["affordability_salary"].mean().sort_values()

median_state=state_affordability.index[len(state_affordability)//2]
REFERENCE_STATE=median_state

year_dummies=pd.get_dummies(model_df["year"], prefix="year")
year_dummies=year_dummies.drop(columns=[f"year_{REFERENCE_YEAR}"])

state_dummies=pd.get_dummies(model_df["state_code_grouped"], prefix="state")
state_dummies=state_dummies.drop(columns=[f"state_{REFERENCE_STATE}"])

model_df=pd.concat([model_df, year_dummies, state_dummies], axis=1)

year_cols_model=list(year_dummies.columns)
state_cols_model=list(state_dummies.columns)

The reference state is the one whose mean purchasing power falls at the median of the state means, which is Texas. Texas is not a cost of living outlier in either direction, and it holds considerable within state diversity, with major metropolitan areas alongside a large number of rural counties, so reading other states as different from Texas gives an interpretable baseline. <a href="#fig-state-afford" class="quarto-xref">Figure 10</a> shows where it sits.

In [27]:
state_df=state_affordability.reset_index()
state_df.columns=["state","mean_pp"]
state_df["state_label"]=state_df["state"].map(state_name_map)
# the pooled thin-state bucket is not a state; excluded from the ranking display below
state_df=state_df[state_df["state"]!="OTHER"].copy()
state_median=float(state_affordability.median())
state_df["comparison"]=state_df["mean_pp"].ge(state_median).map({True:"Above Average", False:"Below Average"})

In [28]:
%%R -i state_df -i state_median -w 8 -h 10 -u in -r 150
state_df$state_label <- factor(state_df$state_label, levels=state_df$state_label)
median_label <- sprintf("Median: $%.0fk", state_median/1000)
n_states <- length(levels(state_df$state_label))
pp_range <- range(state_df$mean_pp)
low_mid  <- (pp_range[1] + state_median) / 2
high_mid <- (state_median + pp_range[2]) / 2

ggplot(state_df, aes(x=mean_pp, y=state_label)) +
  annotate("rect", xmin=-Inf, xmax=state_median, ymin=-Inf, ymax=Inf,
           fill="#C0392B", alpha=0.10) +
  annotate("rect", xmin=state_median, xmax=Inf, ymin=-Inf, ymax=Inf,
           fill=GREEN, alpha=0.10) +
  annotate("text", x=low_mid, y=n_states / 2, label="Lower Purchasing Power",
           color="#C0392B", fontface="bold", size=6, hjust=0.5, vjust=0.5, alpha=0.25) +
  annotate("text", x=high_mid, y=n_states / 2, label="Higher Purchasing Power",
           color=GREEN, fontface="bold", size=6, hjust=0.5, vjust=0.5, alpha=0.25) +
  geom_vline(xintercept=state_median, linetype="dotted", linewidth=0.6, color="black") +
  annotate("text", x=state_median, y=n_states + 1.9, label=median_label,
           size=3, color="black", hjust=0, vjust=0) +
  geom_segment(aes(x=state_median, xend=mean_pp, yend=state_label), linewidth=0.9, alpha=0.5, color="black") +
  geom_point(size=2.6, color="black") +
  scale_x_continuous(labels=label_dollar(scale=1e-3, suffix="k"), breaks=scales::pretty_breaks(n=10)) +
  scale_y_discrete(expand=expansion(add=c(0.6, 2.6))) +
  labs(title="Mean Purchasing Power by State",
       x="Mean Purchasing Power", y=NULL) +
  theme(axis.text.y=element_text(size=7), plot.margin=margin(10,10,10,10))

### Collinearity

The reference group logic of <a href="#fig-vif" class="quarto-xref">Figure 4</a> applies here unchanged, so `non_routine_cognitive_share` is again omitted and the remaining three coefficients read as shifts toward each group and away from it. What is new in this feature set is the indicator blocks. <a href="#fig-vif-ml" class="quarto-xref">Figure 11</a> checks that the year and state indicators introduce no collinearity of their own.

In [29]:
exposure_check_cols=["routine_cognitive_share",
    "routine_manual_share",
    "non_routine_cognitive_share",
    "non_routine_manual_share"]

REFERENCE_EXPOSURE="non_routine_cognitive_share"
exposure_cols_model=[c for c in exposure_check_cols if c!=REFERENCE_EXPOSURE]
core_features=exposure_cols_model+["poverty_rate", "unemployment_rate"]

In [30]:
vif_features=core_features+year_cols_model+state_cols_model

X_vif=sm.add_constant(model_df[vif_features].astype(float))

vif_data=pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
vif_top=vif_data[vif_data["feature"].isin(year_cols_model+state_cols_model)].sort_values("VIF", ascending=False).head(10).copy()
vif_top["Indicator"]=(vif_top["feature"]
                      .str.replace("year_", "", regex=False)
                      .str.replace("state_", "State ", regex=False))
vif_plot_df=vif_top[["Indicator","VIF"]].sort_values("VIF")

In [31]:
%%R -i vif_plot_df -w 8 -h 4 -u in -r 150
vif_plot_df$Indicator <- factor(vif_plot_df$Indicator, levels=vif_plot_df$Indicator)

ggplot(vif_plot_df, aes(x=VIF, y=Indicator)) +
  geom_segment(aes(x=0, xend=VIF, yend=Indicator), linewidth=1.1, color="#C8C8C5") +
  geom_point(size=4, color=BLUE) +
  geom_text(aes(label=sprintf("%.2f", VIF)), hjust=-0.5, size=3.6, color="#252525") +
  geom_vline(xintercept=5, linetype="dashed", linewidth=0.7, color=ORANGE) +
  annotate("text", x=5, y=Inf, label="VIF = 5", hjust=1.1, vjust=1.3, color="grey30", size=3.5) +
  scale_x_continuous(limits=c(0, 5.6), breaks=0:5, expand=c(0,0)) +
  labs(title="Highest Variance Inflation Factors, Year and State Indicators",
       x="Variance Inflation Factor", y="Year")

<a href="#fig-vif-ml" class="quarto-xref">Figure 11</a> lists the highest variance inflation factors after the reference group is removed. No offenders remain, since every feature falls comfortably below the threshold of 5.

In [32]:
# feature matrix, built now that the year and state indicators and the reference
# group logic above are settled; sliced using the grouped split made earlier
feature_cols_reg=core_features+year_cols_model+state_cols_model

X_ml=model_df[feature_cols_reg].astype(float)
y=model_df[target_col]

X_train, X_test=X_ml.iloc[train_idx], X_ml.iloc[test_idx]
y_train, y_test=y.iloc[train_idx], y.iloc[test_idx]

# baseline feature set (no year/state), reusing the SAME row split as the full model
feature_cols_base=exposure_cols_model+["poverty_rate", "unemployment_rate"]
X_base=model_df[feature_cols_base].astype(float)
X_train_base, X_test_base=X_base.iloc[train_idx], X_base.iloc[test_idx]

### Linear regression

This specification is a predictive benchmark, not a second inferential model. It reuses the diagnostic lessons the panel regression above already established, now fit on the training split alone so it is directly comparable to the random forest and neural network that follow. We report two versions of each linear model, estimated by ordinary least squares (OLS). The baseline uses the task groups with poverty and unemployment only, and the full model adds year and state controls. The baseline shows what the relationship looks like before time and geography are accounted for, the full model is the one carried forward, and reporting both makes visible exactly what changes when time and geography enter.

#### Baseline specification

In [33]:
X_train_base_sm=sm.add_constant(X_train_base)
ols_model_base=sm.OLS(y_train, X_train_base_sm).fit()

The baseline reaches an R² of 0.766. Every coefficient is negative, which follows from non-routine cognitive serving as the reference. Shifting a county toward any of the other three groups is associated with lower purchasing power, and non-routine manual is the steepest. The coefficients appear alongside the full specifications in <a href="#tbl-ols-comparison" class="quarto-xref">Table 2</a>.

#### Full specification

In [34]:
X_train_sm=sm.add_constant(X_train)
ols_model=sm.OLS(y_train, X_train_sm).fit()

Adding time and geography raises R² from 0.766 to 0.871, so these variables explain real variance. The `unemployment_rate` coefficient flips sign, which suggests the baseline coefficient absorbed some of the geographic and temporal confounding, and that is what we would expect, since places and periods with heavy unemployment also tend to be places and periods of low purchasing power. The full model’s coefficient reflects the within state year relationship instead. The task group coefficients keep their direction and their relative ordering through the addition.

The raw target residuals show the same systematic curvature already documented in <a href="#fig-diagnostics" class="quarto-xref">Figure 7</a> (Panels A and B), so the linear specification is poorly suited to the untransformed target here as well.

#### Full specification on the logged outcome

The binned means in <a href="#fig-binned-scale" class="quarto-xref">Figure 8</a> showed an approximately exponential pattern, so we test whether logging the target makes a linear model appropriate here as well.

In [35]:
assert (y<=0).sum()==0  # confirm log is safe

y_train_log=np.log(y_train)
y_test_log=np.log(y_test)

ols_model_log=sm.OLS(y_train_log, X_train_sm).fit()

X_test_sm=sm.add_constant(X_test, has_constant="add")
y_pred_ols_log=ols_model_log.predict(X_test_sm)
y_pred_ols_log_dollars=np.exp(y_pred_ols_log)

ols_log_test_r2_log=r2_score(y_test_log, y_pred_ols_log)
ols_log_test_r2=r2_score(y_test, y_pred_ols_log_dollars)
ols_log_test_mae=mean_absolute_error(y_test, y_pred_ols_log_dollars)

Logging the target resolves the pattern here the same way it did for the panel regression in <a href="#fig-diagnostics" class="quarto-xref">Figure 7</a> (Panels C and D): the residual spread evens out and the QQ plot tracks the diagonal more closely, so a linear model is appropriate on the logged target. That model reaches an in sample R² of 0.913 and a held out R², back transformed into dollars, of 0.896.

#### Coefficient comparison

<a href="#tbl-ols-comparison" class="quarto-xref">Table 2</a> reports the coefficients across the three specifications, and <a href="#sec-results" class="quarto-xref">Section 5</a> shows the same comparison graphically.

In [36]:
log_coefs=ols_model_log.params[feature_cols_base]

# task groups are 0 to 1 scale (1pp=0.01 units); poverty and unemployment are already in point units
unit_per_point={
    "routine_cognitive_share": 0.01,
    "routine_manual_share": 0.01,
    "non_routine_manual_share": 0.01,
    "poverty_rate": 1.0,
    "unemployment_rate": 1.0,
}

pct_change_per_point={
    feat: (np.exp(log_coefs[feat]*unit_per_point[feat])-1)*100
    for feat in feature_cols_base
}

label_map={
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_manual_share":"Non-Routine Manual",
    "poverty_rate":"Poverty rate",
    "unemployment_rate":"Unemployment rate",
}

comparison_ols=pd.DataFrame({
    "Variable": [label_map[f] for f in feature_cols_base],
    "Baseline ($)": ols_model_base.params[feature_cols_base].values,
    "With controls ($)": ols_model.params[feature_cols_base].values,
    "With controls (log)": ols_model_log.params[feature_cols_base].values,
    "Percent change per point": [pct_change_per_point[f] for f in feature_cols_base],
})

style_table(GT(comparison_ols)
  .tab_header(title="Coefficients Across the Three OLS Specifications")
  .fmt_number(columns=["Baseline ($)","With controls ($)"], decimals=0, use_seps=True)
  .fmt_number(columns="With controls (log)", decimals=4)
  .fmt_number(columns="Percent change per point", decimals=2)
  .tab_source_note("Task group coefficients are per unit of group proportion."))

The sign flip in `unemployment_rate` holds across all three fitted forms. Task group coefficients shrink in magnitude from baseline to raw controls while the poverty coefficient grows, from roughly −1,376 to −1,729, so the controls sharpen the task group association rather than displacing it.

Each additional percentage point of poverty rate is associated with roughly 2.9 percent lower purchasing power, holding time, geography, and task groups fixed. That is a considerably larger proportional difference than any single task group carries.

In [37]:
spec_summary = pd.DataFrame({
    "Metric": ["Outcome", "Year + state FE", "R²", "Residual diagnostics", "Retained for inference"],
    "Baseline": ["Purchasing power", "No", f"{ols_model_base.rsquared:.3f}", "—", "No"],
    "Full, levels": ["Purchasing power", "Yes", f"{ols_model.rsquared:.3f}", "Failed", "No"],
    "Full, log": ["log(Purchasing power)", "Yes", f"{ols_model_log.rsquared:.3f}", "Passed", "Yes"],
})

In [38]:
style_table(GT(spec_summary, rowname_col="Metric")
  .tab_header(title="Model Specification Comparison")
  .tab_style(style=style.text(weight="bold"),
             locations=loc.body(columns="Full, log", rows=[2, 3, 4])))

<a href="#tbl-specmatrix" class="quarto-xref">Table 3</a> shows in sample R² climbing through each stage. Adding year and state controls raises it from 0.766 to 0.871, confirming that time and geography explain real variance the baseline left unmodeled. Logging the target raises it further, to 0.913, and resolves the residual violations along the way. This is the linear model used for inference from here, since the raw target model’s coefficient significance cannot be trusted given its failed diagnostics.

### Random forest

For the random forest we ran a grid search over the number of trees (600 and 1,000), maximum depth (10, 20, and unlimited), minimum leaf size (1, 2, and 5), and the feature subsampling rate (all features, half, and the square root), scored by grouped five fold cross validation on the training counties. More trees reduce variance, while depth and leaf size supply regularization. The search selected 1,000 trees, unlimited depth, a minimum leaf of 1, and half the features per split.

In [39]:
# grid search actually run once; kept for documentation of the search space,
# not re-executed on render. Best params hardcoded in the following chunk.

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold

rf=RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid={
    "n_estimators": [600, 1000],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 5],
    "max_features": [1.0, "sqrt", 0.5],
}

group_cv=GroupKFold(n_splits=5)
gcv=GridSearchCV(rf, param_grid, cv=group_cv, scoring="r2", n_jobs=-1)
gcv.fit(X_train, y_train, groups=groups_train)

In [40]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

# grid search above was run once over the full param_grid; best params found were:
# {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 1000}
best_params={
    "max_depth": None,
    "max_features": 0.5,
    "min_samples_leaf": 1,
    "n_estimators": 1000,
}

best_rf=RandomForestRegressor(**best_params, random_state=42, n_jobs=2)
best_rf.fit(X_train, y_train)

y_pred_rf=best_rf.predict(X_test)
rf_test_r2=r2_score(y_test, y_pred_rf)
rf_test_mae=mean_absolute_error(y_test, y_pred_rf)
rf_train_r2=best_rf.score(X_train, y_train)

The forest reaches a held out R² of 0.876 on the raw target. The log target OLS model’s held out figure of 0.896 is the relevant comparison, since both are scored on the same held out counties, and the forest does not improve on it.

The forest fits the training data far more closely than it fits unseen counties, 0.989 against 0.876. Its additional flexibility therefore captures patterns in the training sample that do not carry over to counties it has not seen.

In [41]:
rf_resid_df=pd.DataFrame({"pred":y_pred_rf, "resid":y_test-y_pred_rf})

In [42]:
%%R -i rf_resid_df -w 7 -h 5 -u in -r 150
ggplot(rf_resid_df, aes(x=pred, y=resid)) +
  geom_point(alpha=0.3, size=0.6, color=PINK) +
  geom_hline(yintercept=0, linetype="dashed", linewidth=0.4) +
  scale_x_continuous(labels=dollar_axis) +
  scale_y_continuous(labels=dollar_axis) +
  labs(title="Random Forest Residuals vs. Predicted Values",
       x="Predicted Values", y="Residuals")

Unlike the raw target OLS residuals, <a href="#fig-rf-resid" class="quarto-xref">Figure 12</a> shows no systematic curve or trend across the range of predicted values.

In [43]:
perm_imp=permutation_importance(
    best_rf, X_test, y_test, n_repeats=20, random_state=42, n_jobs=1
)

#### Random forest specification and ablation

In [44]:
rf_log=RandomForestRegressor(**best_params, random_state=42, n_jobs=2)
rf_log.fit(X_train, y_train_log)

y_pred_rf_log=rf_log.predict(X_test)
y_pred_rf_log_dollars=np.exp(y_pred_rf_log)

rf_log_test_r2_log=r2_score(y_test_log, y_pred_rf_log)
rf_log_test_r2=r2_score(y_test, y_pred_rf_log_dollars)
rf_log_test_mae=mean_absolute_error(y_test, y_pred_rf_log_dollars)

The random forest is largely insensitive to whether purchasing power is modeled on its original or logarithmic scale. Fitting the model to the logged outcome and back transforming predictions gives an R² of 0.877, compared with 0.876 when the outcome is modeled directly. Because the log specification is also used for the primary linear model, the log target forest is used for the comparisons that follow.

In [45]:
feature_cols_no_exposure=[c for c in feature_cols_reg if c not in exposure_cols_model]
X_train_ne=X_train[feature_cols_no_exposure]
X_test_ne=X_test[feature_cols_no_exposure]

rf_no_exposure=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_no_exposure.fit(X_train_ne, y_train_log)
rf_no_exposure_r2=r2_score(y_test, np.exp(rf_no_exposure.predict(X_test_ne)))

Removing the task groups reduces held out R² from 0.877 to 0.834, a decline of 0.043.

In [46]:
feature_cols_no_econ=[c for c in feature_cols_reg if c not in ["poverty_rate", "unemployment_rate"]]
X_train_ne2=X_train[feature_cols_no_econ]
X_test_ne2=X_test[feature_cols_no_econ]

rf_no_econ=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_no_econ.fit(X_train_ne2, y_train_log)
rf_no_econ_r2=r2_score(y_test, np.exp(rf_no_econ.predict(X_test_ne2)))

By comparison, a model with the task groups but without poverty and unemployment reaches an R² of 0.703. Task groups therefore carry substantial predictive information on their own, but the strongest performance comes from combining them with the economic controls. All three models use the same forest specification, logged outcome, and held out counties, so their performance is directly comparable.

#### Grouped permutation importance

Ablation measures how well the model compensates when a whole set of predictors is removed and it is refit from scratch. To see how strongly the fitted model actually relies on the task groups, we instead permute the modeled task group proportions jointly and measure the resulting drop in R², rather than removing and refitting.

In [47]:
exposure_cols=exposure_cols_model  # the three non-reference task groups
rng=np.random.RandomState(42)

baseline_r2=r2_score(y_test, best_rf.predict(X_test))

def grouped_permutation_drop(cols, n_repeats=20):
    drops=[]
    for _ in range(n_repeats):
        X_perm=X_test.copy()
        shuffled_idx=rng.permutation(len(X_perm))
        X_perm[cols]=X_perm[cols].values[shuffled_idx]
        drops.append(baseline_r2-r2_score(y_test, best_rf.predict(X_perm)))
    return np.mean(drops)

joint_drop=grouped_permutation_drop(exposure_cols)

individual_drops={}
for col in exposure_cols:
    individual_drops[col]=grouped_permutation_drop([col])

year_drop=grouped_permutation_drop(year_cols_model)
state_drop=grouped_permutation_drop(state_cols_model)

Jointly permuting the task group proportions drops R² by 0.153, well beyond the 0.043 decline from removing them and refitting. The gap points to a real difference between the two methods. When the task groups are dropped before training, the forest can recover part of their signal from correlated predictors such as poverty and time. Permutation instead disrupts that information after the model has already learned to rely on it, so no such recovery is possible. The task groups therefore overlap with poverty, unemployment, and time while still adding something those three do not capture on their own.

Year produces a substantially larger permutation decline than state. The drop is 0.128 for year against 0.008 for state, conditional on the economic controls and task groups already in the model, so time carries far more of the model’s remaining predictive signal than geography does.

Among the three modeled proportions, `non_routine_manual_share` shows the largest individual permutation effect. None of the three should be read as additive or independent contributions, since the task groups are compositional and mechanically related to one another. This ranking is consistent with the panel regression, where non-routine manual carries the steepest coefficient among the task groups.

### Neural network

The neural network is kept shallow, with three hidden layers and a single linear output unit for the continuous outcome.

In [48]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)

scaler_x=StandardScaler().fit(X_train)
X_train_s=scaler_x.transform(X_train)
X_test_s=scaler_x.transform(X_test)

scaler_y=StandardScaler().fit(y_train.values.reshape(-1, 1))
y_train_s=scaler_y.transform(y_train.values.reshape(-1, 1)).ravel()
y_test_s=scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

n_features=X_train.shape[1]
h1=int(round(n_features*1.5))
h2=max(int(round(h1*0.5)), 20)
h3=10

nn=models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(h1, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(h2, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(h3, activation="relu"),
    layers.Dense(1, activation="linear"),  # regression head
])

nn.compile(optimizer="adam", loss="mse", metrics=["mae"])

es=callbacks.EarlyStopping(patience=15, restore_best_weights=True)

history=nn.fit(
    X_train_s, y_train_s,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[es],
    verbose=0,
)

y_pred_nn_s=nn.predict(X_test_s).ravel()
y_pred_nn=scaler_y.inverse_transform(y_pred_nn_s.reshape(-1, 1)).ravel()

nn_initial_test_r2=r2_score(y_test, y_pred_nn)
nn_initial_test_mae=mean_absolute_error(y_test, y_pred_nn)

nn_init_df=pd.DataFrame({
    "epoch": range(1, len(history.history["loss"])+1),
    "Train": history.history["loss"],
    "Validation": history.history["val_loss"],
}).melt(id_vars="epoch", var_name="series", value_name="loss")

Training loss falls steadily while validation loss reverses direction partway through, which is overfitting; <a href="#fig-nn-curves" class="quarto-xref">Figure 13</a> panel A shows this, alongside the regularized model in panel B. We respond with L2 weight regularization, heavier dropout, and a shorter early stopping patience.

In [49]:
from tensorflow.keras import regularizers

nn=models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(h1, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(h2, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(h3, activation="relu"),
    layers.Dense(1, activation="linear"),
])

nn.compile(optimizer="adam", loss="mse", metrics=["mae"])

es=callbacks.EarlyStopping(patience=8, restore_best_weights=True)

history=nn.fit(
    X_train_s, y_train_s,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[es],
    verbose=0,
)

y_pred_nn_s=nn.predict(X_test_s).ravel()
y_pred_nn=scaler_y.inverse_transform(y_pred_nn_s.reshape(-1, 1)).ravel()

nn_test_r2=r2_score(y_test, y_pred_nn)
nn_test_mae=mean_absolute_error(y_test, y_pred_nn)

nn_reg_df=pd.DataFrame({
    "epoch": range(1, len(history.history["loss"])+1),
    "Train": history.history["loss"],
    "Validation": history.history["val_loss"],
}).melt(id_vars="epoch", var_name="series", value_name="loss")

In [50]:
%%R -i nn_init_df -i nn_reg_df -w 10 -h 4.5 -u in -r 150
ymax <- max(c(nn_init_df$loss, nn_reg_df$loss)) * 1.05
y_breaks <- seq(0, 0.8, by=0.2)

best_init <- nn_init_df[nn_init_df$series=="Validation",]
best_init_epoch <- best_init$epoch[which.min(best_init$loss)]
best_init_loss <- min(best_init$loss)
best_reg <- nn_reg_df[nn_reg_df$series=="Validation",]
best_reg_epoch <- best_reg$epoch[which.min(best_reg$loss)]
best_reg_loss <- min(best_reg$loss)

label_df1 <- nn_init_df[nn_init_df$epoch==max(nn_init_df$epoch),]
label_df2 <- nn_reg_df[nn_reg_df$epoch==max(nn_reg_df$epoch),]
label_nudge <- ymax*0.018
label_df1$label_y <- label_df1$loss + ifelse(label_df1$series=="Train", -label_nudge, label_nudge)
label_df2$label_y <- label_df2$loss + ifelse(label_df2$series=="Train", -label_nudge, label_nudge)

p1 <- ggplot(nn_init_df, aes(x=epoch, y=loss, color=series)) +
  geom_segment(aes(x=best_init_epoch, xend=best_init_epoch, y=0, yend=best_init_loss + ymax*0.08),
               inherit.aes=FALSE, linetype="dashed", linewidth=0.35, color="black") +
  geom_line(linewidth=0.8) +
  geom_point(data=best_init[best_init$epoch==best_init_epoch,], aes(x=epoch, y=loss),
             inherit.aes=FALSE, color=ORANGE, size=2.2) +
  geom_label(data=label_df1, aes(label=series, y=label_y), hjust=-0.1, size=3.4, fontface="bold",
             fill="#FCFCFB", linewidth=0, label.padding=unit(0.08, "lines"), show.legend=FALSE) +
  annotate("label", x=best_init_epoch, y=best_init_loss + ymax*0.1,
           label=paste0("Minimum validation loss · epoch ", best_init_epoch),
           hjust=0, size=2.9, color="grey35", fill="#FCFCFB", linewidth=0,
           label.padding=unit(0.12, "lines")) +
  scale_color_manual(values=c(Train=BLUE, Validation=ORANGE), guide="none") +
  scale_y_continuous(limits=c(0, ymax), breaks=y_breaks) +
  scale_x_continuous(expand=expansion(mult=c(0.02, 0.28))) +
  labs(x="Epoch", y="Loss (MSE, Standardized)", title="Initial Model",
       subtitle="Patience 15")

p2 <- ggplot(nn_reg_df, aes(x=epoch, y=loss, color=series)) +
  geom_segment(aes(x=best_reg_epoch, xend=best_reg_epoch, y=0, yend=best_reg_loss + ymax*0.08),
               inherit.aes=FALSE, linetype="dashed", linewidth=0.35, color="black") +
  geom_line(linewidth=0.8) +
  geom_point(data=best_reg[best_reg$epoch==best_reg_epoch,], aes(x=epoch, y=loss),
             inherit.aes=FALSE, color=ORANGE, size=2.2) +
  geom_label(data=label_df2, aes(label=series, y=label_y), hjust=-0.1, size=3.4, fontface="bold",
             fill="#FCFCFB", linewidth=0, label.padding=unit(0.08, "lines"), show.legend=FALSE) +
  annotate("label", x=best_reg_epoch, y=best_reg_loss + ymax*0.1,
           label=paste0("Minimum validation loss · epoch ", best_reg_epoch),
           hjust=0, size=2.9, color="grey35", fill="#FCFCFB", linewidth=0,
           label.padding=unit(0.12, "lines")) +
  scale_color_manual(values=c(Train=BLUE, Validation=ORANGE), guide="none") +
  scale_y_continuous(limits=c(0, ymax), breaks=y_breaks) +
  scale_x_continuous(expand=expansion(mult=c(0.02, 0.28))) +
  labs(x="Epoch", y=NULL, title="Regularized Model",
       subtitle="L2 regularization, patience 8")

p1 + p2 +
  plot_annotation(title="Training and Validation Loss Across Model Specifications",
                  tag_levels="A",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1:   
R callback write-console: In geom_segment(aes(x = best_init_epoch, xend = best_init_epoch,  :  
R callback write-console: 
   
R callback write-console:  All aesthetics have length 1, but the data has 38 rows.
ℹ Please consider using `annotate()` or provide this layer with data containing
  a single row.
  
R callback write-console: 2:   
R callback write-console: In geom_segment(aes(x = best_reg_epoch, xend = best_reg_epoch, y = 0,  :  
R callback write-console: 
   
R callback write-console:  All aesthetics have length 1, but the data has 108 rows.
ℹ Please consider using `annotate()` or provide this layer with data containing
  a single row.
  

The gap narrows after regularization, but the network still does not beat the random forest or the log target linear model, both of which are far cheaper to fit and tune. This project is built to explain the relationship between task groups and purchasing power rather than to maximize predictive accuracy, so we stop tuning here and carry the linear and forest models forward as the primary results.

### Cross validation

Having compared explained variance across the three models, we cross validate to check that the single split result generalizes.

In [51]:
from sklearn.model_selection import GroupKFold

group_kfold_cv=GroupKFold(n_splits=5)
cv_results={
    "Ordinary Least Squares": [],
    "Random forest": [],
}

y_log=np.log(y)  # full data log target, same idea as y_train_log/y_test_log

for fold, (tr_idx, val_idx) in enumerate(group_kfold_cv.split(X_ml, y, groups)):
    X_tr, X_val=X_ml.iloc[tr_idx], X_ml.iloc[val_idx]
    y_tr, y_val=y.iloc[tr_idx], y.iloc[val_idx]
    y_tr_log=y_log.iloc[tr_idx]

    X_tr_sm=sm.add_constant(X_tr)
    X_val_sm=sm.add_constant(X_val, has_constant="add")
    ols_fold_log=sm.OLS(y_tr_log, X_tr_sm).fit()
    pred_ols=np.exp(ols_fold_log.predict(X_val_sm))  # back transformed, dollar scale
    cv_results["Ordinary Least Squares"].append(r2_score(y_val, pred_ols))

    rf_fold=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
    rf_fold.fit(X_tr, y_tr_log)
    pred_rf=np.exp(rf_fold.predict(X_val))
    cv_results["Random forest"].append(r2_score(y_val, pred_rf))

cv_folds=pd.DataFrame([
    {"model": m, "fold": i+1, "r2": s}
    for m, scores in cv_results.items()
    for i, s in enumerate(scores)
])

cv_summary=pd.DataFrame({m: {"Mean R²": np.mean(s), "SD": np.std(s)}
                         for m, s in cv_results.items()}).T.reset_index()
cv_summary.columns=["Model","Mean R²","SD"]

In [52]:
style_table(GT(cv_summary)
  .tab_header(title="Five Fold Grouped Cross Validation Performance")
  .fmt_number(columns=["Mean R²","SD"], decimals=4)
  .cols_align(align="right", columns=["Mean R²","SD"])
  .tab_source_note("Both models are fit on the log target and scored on the dollar scale after back transformation."))

<a href="#tbl-cv-summary" class="quarto-xref">Table 4</a> shows the two models performing at the same level across five folds. The difference between their means is smaller than the variation across folds, so the models perform similarly relative to that variation. The single split result above is therefore not an artifact of one particular split, and both models generalize comparably across held out counties.

The neural network is not included, since cross validating it would be expensive and it is not carried forward.

<a href="#sec-results" class="quarto-xref">Section 5</a> reports the model comparison across held out R² and mean absolute error.

## Summary

Three specifications answer three different questions. The panel regression, with standard errors clustered by county, is the model used for inference. It shows that routine intensive work is associated with lower purchasing power even after controlling for poverty, unemployment, population, and year, and the variance decomposition establishes that the task group differences behind that association are durable features of a place rather than transient ones. Its log respecification resolves the residual violations that make the raw target version untrustworthy for that purpose.

The predictive models answer a different question, which is how much of purchasing power can be explained at all, and by what. The random forest and the log target linear model perform comparably on held out data, so the curvature in the relationship adds little once poverty, unemployment, time, and geography are already in the model. The neural network improves on neither, and its added complexity is not justified by this feature set.

Across both approaches the task groups carry real, independent explanatory power, while poverty rate remains the dominant single factor. That ordering, rather than any one model’s fit statistic, is the finding this analysis is built to support, and it is what <a href="#sec-results" class="quarto-xref">Section 5</a> carries forward.

# Results

In [53]:
# inbound from _04: reg, df, model, model_log, X_ml, X_test, y_test, y_pred_rf,
# y_pred_rf_log_dollars, best_rf, perm_imp, joint_drop, year_drop, state_drop,
# cv_folds, engine, ols_log_test_r2, rf_log_test_r2, nn_test_r2,
# ols_log_test_mae, rf_log_test_mae, nn_test_mae
import pandas as pd
import numpy as np
import statsmodels.api as sm
from great_tables import GT, html

res_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share",
          "poverty_rate","unemployment_rate","log_population"]

mean_pp=float(reg["affordability_salary"].mean())
group_terms=["non_routine_manual_share","routine_cognitive_share","routine_manual_share"]
group_names=["Non-Routine Manual","Routine Cognitive","Routine Manual"]

The analysis supports four findings. A county’s task groups are associated with its purchasing power, and the association survives controls for poverty, unemployment, population, and year. The log specification captures most of the structure in the relationship, and the flexible models do not improve on it, which is what supports reading the relationship as proportional rather than fixed in dollars. Poverty rate remains the single strongest correlate of purchasing power throughout, with the task groups adding real signal beyond it. And the task group differences behind the association are durable features of places, so the purchasing power gaps they carry are durable too. The subsections below work through these findings, and through the geography and the model behavior behind them.

## The association

In [54]:
# log specification coefficients (model_log fit in _04, cluster robust), expressed as
# dollars at the panel mean per one percentage point shift
coef_rows=[]
for term, name in zip(group_terms, group_names):
    b=model_log.params[term]
    se=model_log.bse[term]
    est=(np.exp(b*0.01)-1)*mean_pp
    lo=(np.exp((b-1.96*se)*0.01)-1)*mean_pp
    hi=(np.exp((b+1.96*se)*0.01)-1)*mean_pp
    coef_rows.append({"group":name, "est":est, "lo":lo, "hi":hi})
coef_df=pd.DataFrame(coef_rows)

The log specification of the panel regression is the model we use for inference, for the diagnostic reasons set out in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. <a href="#fig-coefplot" class="quarto-xref">Figure 14</a> reports its task group coefficients as the change in purchasing power, in dollars at the panel mean, associated with a one percentage point shift into each group and out of non-routine cognitive work, the reference group. A one percentage point shift toward non-routine manual work is associated with roughly \$846 less in purchasing power, with a 95 percent interval of plus or minus \$38, evaluated at the panel mean. Routine cognitive and routine manual shifts carry smaller but clearly negative associations, and the ordering matches the level specification reported in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. For scale, each additional percentage point of poverty rate is associated with roughly 2.9 percent lower purchasing power, a larger proportional difference than any single task group carries.

In [55]:
%%R -i coef_df -w 8 -h 3.5 -u in -r 150
coef_df$group <- factor(coef_df$group,
    levels=rev(c("Non-Routine Manual","Routine Cognitive","Routine Manual")))

ggplot(coef_df, aes(x=est, y=group, color=group)) +
  geom_vline(xintercept=0, linetype="dashed", linewidth=0.4) +
  geom_errorbar(aes(xmin=lo, xmax=hi), orientation="y", width=0.15) +
  geom_point(size=2.6) +
  geom_text(aes(label=dollar_axis(est)), vjust=-1.4, size=3.3, fontface="bold", show.legend=FALSE) +
  scale_color_manual(values=TASK_COLORS, guide="none") +
  scale_x_continuous(labels=dollar_axis, expand=expansion(mult=c(0.1, 0.15))) +
  labs(title="Task Group Coefficients, Log Specification",
       x="Change in Purchasing Power per Percentage Point Shift (at Panel Mean)", y=NULL)

## Not poverty in disguise

A natural objection is that the task groups merely proxy for poverty, since poor counties hold more routine work. <a href="#fig-baseline-maps" class="quarto-xref">Figure 15</a> shows the two conventional distress measures side by side, mean poverty rate (Panel A) and mean unemployment rate (Panel B) by county across the study period. Both run highest across the rural South, the Mississippi Delta, and pockets of the Southwest border region. That is the same broad area where <a href="#fig-afford-map" class="quarto-xref">Figure 17</a> shows purchasing power running lowest, which is exactly why the objection is worth taking seriously. Unemployment (Panel B) carries a second concentration along the California coast and Central Valley that poverty (Panel A) does not share, evidence the two measures are not interchangeable with each other or with what follows.

In [56]:
import geopandas as gpd

map_baseline=pd.read_sql("""
    select county_fips, avg(poverty_rate) as poverty_rate, avg(unemployment_rate) as unemployment_rate
    from county_baseline
    group by county_fips
""", engine)
map_baseline['fips']=map_baseline['county_fips'].astype(str).str.zfill(5)

gdf_baseline=gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip")
gdf_baseline=gdf_baseline.rename(columns={'GEOID':'fips'})
merged_baseline=gdf_baseline.merge(map_baseline[['fips','poverty_rate','unemployment_rate']], on='fips', how='left')
merged_baseline=merged_baseline[~merged_baseline['STATEFP'].isin(['02','15','60','66','69','72','78'])]
os.makedirs("output", exist_ok=True)
merged_baseline[['fips','poverty_rate','unemployment_rate','geometry']].to_file("output/baseline_maps.geojson", driver="GeoJSON")

pov_vmin=float(merged_baseline['poverty_rate'].quantile(0.02))
pov_vmax=float(merged_baseline['poverty_rate'].quantile(0.98))
unemp_vmin=float(merged_baseline['unemployment_rate'].quantile(0.02))
unemp_vmax=float(merged_baseline['unemployment_rate'].quantile(0.98))

In [57]:
%%R -i pov_vmin -i pov_vmax -i unemp_vmin -i unemp_vmax -w 11 -h 5.5 -u in -r 150
suppressMessages(library(sf))

baseline_sf <- st_read("output/baseline_maps.geojson", quiet=TRUE)

p1 <- ggplot(baseline_sf) +
  geom_sf(aes(fill=poverty_rate), color="white", linewidth=0.05) +
  scale_fill_distiller(palette="YlOrBr", direction=1, limits=c(pov_vmin, pov_vmax), oob=scales::squish,
                        na.value="#eeeeee", name="Poverty Rate (%)") +
  coord_sf(crs=st_crs(5070)) +
  theme_void() +
  theme(legend.position="bottom", legend.key.width=unit(1.2, "cm"),
        plot.title=element_text(face="bold", size=11, hjust=0.5),
        plot.background=element_rect(fill="#FCFCFB", color=NA)) +
  labs(title="Poverty Rate")

p2 <- ggplot(baseline_sf) +
  geom_sf(aes(fill=unemployment_rate), color="white", linewidth=0.05) +
  scale_fill_distiller(palette="YlOrBr", direction=1, limits=c(unemp_vmin, unemp_vmax), oob=scales::squish,
                        na.value="#eeeeee", name="Unemployment Rate (%)") +
  coord_sf(crs=st_crs(5070)) +
  theme_void() +
  theme(legend.position="bottom", legend.key.width=unit(1.2, "cm"),
        plot.title=element_text(face="bold", size=11, hjust=0.5),
        plot.background=element_rect(fill="#FCFCFB", color=NA)) +
  labs(title="Unemployment Rate")

p1 + p2 +
  plot_annotation(title="Two Conventional Distress Measures, by County",
                  tag_levels="A",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

<a href="#fig-coefcompare" class="quarto-xref">Figure 16</a> tests the objection directly by fitting the log panel regression twice on the same rows. The first specification includes only task groups and year indicators; the second adds poverty rate, unemployment rate, and log population, the specification reported in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. Non-routine manual and routine manual both shrink substantially when the controls enter, which is what we would expect, since poverty absorbs shared variation. Routine cognitive barely moves, at roughly −1.1 percent either way, evidence its association runs through something other than the poverty and unemployment channel the controls capture. All three remain negative and precisely estimated after the controls enter. The task group association is smaller than it first appears for two of the three groups and clearly real for all of them.

In [58]:
share_terms=["routine_cognitive_share","routine_manual_share","non_routine_manual_share"]

X_nocontrols=pd.concat([reg[share_terms],
                        pd.get_dummies(reg["year"], prefix="year", drop_first=True).astype(float)], axis=1)
X_nocontrols=sm.add_constant(X_nocontrols)
model_nc_log=sm.OLS(reg["actual_log"], X_nocontrols).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})

def pct_ci(m, term):
    b=m.params[term]
    se=m.bse[term]
    est=(np.exp(b*0.01)-1)*100
    lo=(np.exp((b-1.96*se)*0.01)-1)*100
    hi=(np.exp((b+1.96*se)*0.01)-1)*100
    return est, lo, hi

compare_rows=[]
for term, name in zip(share_terms, ["Routine Cognitive","Routine Manual","Non-Routine Manual"]):
    for m, spec in [(model_nc_log, "Task Groups and Year Only"), (model_log, "With Controls")]:
        est, lo, hi = pct_ci(m, term)
        compare_rows.append({"group":name, "spec":spec, "est":est, "lo":lo, "hi":hi})
compare_df=pd.DataFrame(compare_rows)

compare_wide=compare_df.pivot(index="group", columns="spec", values="est").reset_index()
compare_wide.columns=["group","est_baseline","est_controls"]
compare_wide["attenuation_pct"]=100*(1-compare_wide["est_controls"].abs()/compare_wide["est_baseline"].abs())
compare_wide["attenuation_label"]=compare_wide["attenuation_pct"].apply(
    lambda v: "Little change" if v<5 else f"{v:.0f}% smaller")

In [59]:
%%R -i compare_df -i compare_wide -w 9 -h 4.5 -u in -r 150
compare_df$group <- factor(compare_df$group,
    levels=rev(c("Non-Routine Manual","Routine Cognitive","Routine Manual")))
compare_df$spec <- factor(compare_df$spec,
    levels=c("Task Groups and Year Only","With Controls"))
compare_wide$group <- factor(compare_wide$group,
    levels=rev(c("Non-Routine Manual","Routine Cognitive","Routine Manual")))
compare_df$label_nudge <- ifelse(compare_df$spec=="With Controls", 0.35, -0.35)
compare_wide$mid <- (compare_wide$est_baseline+compare_wide$est_controls)/2

ggplot(compare_df, aes(x=est, y=group)) +
  geom_segment(data=compare_wide, aes(x=est_baseline, xend=est_controls, y=group, yend=group),
               inherit.aes=FALSE, color="grey75", linewidth=2.0, lineend="round") +
  geom_errorbar(aes(xmin=lo, xmax=hi, color=spec), orientation="y", width=0.1, linewidth=0.6) +
  geom_point(aes(color=spec, size=spec)) +
  geom_text(aes(label=sprintf("%.1f%%", est), y=as.numeric(group)+label_nudge),
            size=3.3, show.legend=FALSE, fontface="bold", color="#252525") +
  geom_text(data=compare_wide, aes(x=mid, y=as.numeric(group)-0.55, label=attenuation_label),
            inherit.aes=FALSE, color="grey40", size=3.0, fontface="italic") +
  scale_size_manual(values=c("Task Groups and Year Only"=2.6, "With Controls"=3.6), guide="none") +
  geom_vline(xintercept=0, linetype="dashed", linewidth=0.4) +
  scale_color_manual(values=c("Task Groups and Year Only"=ORANGE, "With Controls"=GREEN)) +
  scale_y_discrete(expand=expansion(add=0.7)) +
  labs(title="Task Group Coefficients Before and After Controls",
       x="Percent Change in Purchasing Power per Percentage Point Shift", y=NULL, color=NULL) +
  theme(legend.position="bottom")

## Where purchasing power is strained

The association also has a clear geographic pattern. <a href="#fig-afford-map" class="quarto-xref">Figure 17</a> maps mean purchasing power by county across the study period, using all counties with a purchasing power value rather than the analytical panel alone, since the outcome requires only income and a price parity. Purchasing power runs highest along the metropolitan Northeast corridor, across parts of the upper Midwest, and in pockets of the mountain West, while the lowest values concentrate across the rural South and the southern border region. Panel B relates this geography to local task groups by grouping counties according to whichever task group holds the largest share of local employment. The 664 counties where non-routine cognitive work dominates reach both higher and more varied purchasing power than the 97 dominated by non-routine manual work or the 87 dominated by routine manual work. Both of those smaller groups cluster tightly at the low end. No county in the panel has routine cognitive work as its largest group, so that group does not appear. Panel A shows the outcome side of the association the coefficients estimate. Panel B shows the same association holding at the level of a county’s single largest task group.

In [60]:
import geopandas as gpd

map_all=pd.read_sql("""
    select county_fips, avg(affordability_salary) as purchasing_power
    from county_affordability
    group by county_fips
""", engine)
map_all['fips']=map_all['county_fips'].astype(str).str.zfill(5)

gdf_map=gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip")
gdf_map=gdf_map.rename(columns={'GEOID':'fips'})

afford_vmin=float(map_all['purchasing_power'].quantile(0.02))
afford_vmax=float(map_all['purchasing_power'].quantile(0.98))

merged_map=gdf_map.merge(map_all[['fips','purchasing_power']], on='fips', how='left')
merged_map=merged_map[~merged_map['STATEFP'].isin(['02','15','60','66','69','72','78'])]
os.makedirs("output", exist_ok=True)
merged_map[['fips','purchasing_power','geometry']].to_file("output/afford_map.geojson", driver="GeoJSON")

# dominant task group per county: whichever of the four groups holds the largest mean share
dominant_share_cols=["routine_cognitive_share","routine_manual_share",
                     "non_routine_cognitive_share","non_routine_manual_share"]
dominant_label_map={"routine_cognitive_share":"Routine Cognitive","routine_manual_share":"Routine Manual",
                    "non_routine_cognitive_share":"Non-Routine Cognitive","non_routine_manual_share":"Non-Routine Manual"}
dominant_df=pd.read_sql(f"""
    select cte.county_fips,
           avg(cte.routine_cognitive_share) as routine_cognitive_share,
           avg(cte.routine_manual_share) as routine_manual_share,
           avg(cte.non_routine_cognitive_share) as non_routine_cognitive_share,
           avg(cte.non_routine_manual_share) as non_routine_manual_share,
           avg(ca.affordability_salary) as purchasing_power
    from county_task_exposure cte
    join county_affordability ca on cte.county_fips=ca.county_fips and cte.year=ca.year
    group by cte.county_fips
""", engine)
dominant_df["dominant_group"]=dominant_df[dominant_share_cols].idxmax(axis=1).map(dominant_label_map)

In [61]:
%%R -i afford_vmin -i afford_vmax -i dominant_df -w 9 -h 9 -u in -r 150
suppressMessages(library(sf))

afford_sf <- st_read("output/afford_map.geojson", quiet=TRUE)

DOMINANT_COLORS <- TASK_COLORS[c("Non-Routine Cognitive","Non-Routine Manual","Routine Manual")]

p1 <- ggplot(afford_sf) +
  geom_sf(aes(fill=purchasing_power), color="#FAFAF8", linewidth=0.05) +
  scale_fill_viridis_c(limits=c(afford_vmin, afford_vmax),
                        oob=scales::squish, na.value="#D9D9D6",
                        labels=label_dollar(scale=1e-3, suffix="k"),
                        name="Mean Purchasing Power") +
  coord_sf(crs=st_crs(5070), datum=NA) +
  guides(fill=guide_colorbar(direction="horizontal", title.position="top", title.hjust=0.5,
                             barwidth=unit(4.5,"cm"), barheight=unit(0.35,"cm"))) +
  theme_void() +
  theme(legend.position="bottom",
        legend.title=element_text(size=10, face="bold", color="#333333"),
        legend.text=element_text(size=9, color="#555555"),
        plot.background=element_rect(fill="#FCFCFB", color=NA))

p2 <- ggplot(dominant_df, aes(x=purchasing_power, fill=dominant_group, color=dominant_group)) +
  geom_density(alpha=0.35, linewidth=0.7) +
  scale_fill_manual(values=DOMINANT_COLORS) +
  scale_color_manual(values=DOMINANT_COLORS) +
  scale_x_continuous(labels=dollar_axis) +
  labs(title="Purchasing Power by Dominant Task Group",
       x="Mean Purchasing Power", y="Density", fill=NULL, color=NULL)

(p1 / p2) + plot_layout(heights=c(2.2, 1)) +
  plot_annotation(title="Where Purchasing Power Is Strained",
                  tag_levels="A",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

## The relationship is proportional

<a href="#sec-analysis" class="quarto-xref">Section 4</a> climbed the model complexity ladder and reported where each rung landed. <a href="#fig-modelcomp" class="quarto-xref">Figure 18</a> collects that ending: held out R² across the five grouped cross validation folds for the two models carried forward. Across those folds, the log target linear model and the random forest perform similarly relative to the variation observed, and neither pulls away from the other. <a href="#tbl-modelmetrics" class="quarto-xref">Table 5</a> adds a further comparison to that picture, a single split that also includes the neural network.

In [62]:
cv_means=cv_folds.groupby("model", as_index=False)["r2"].mean().rename(columns={"r2":"mean_r2"})

In [63]:
%%R -i cv_folds -i cv_means -w 8 -h 4 -u in -r 150
cv_means$mean_label <- sprintf("%.3f", cv_means$mean_r2)
cv_means$label_x <- ifelse(cv_means$model=="Ordinary Least Squares", 0.74, 2.26)

ggplot(cv_folds, aes(x=model, y=r2, color=model)) +
  geom_line(aes(x=model, y=r2, group=fold), inherit.aes=FALSE, color="grey75", linewidth=0.5) +
  geom_point(size=2.6) +
  geom_crossbar(data=cv_means, aes(x=model, y=mean_r2, ymin=mean_r2, ymax=mean_r2),
                width=0.35, linewidth=0.5, color="grey30", inherit.aes=FALSE) +
  geom_text(data=cv_means, aes(x=label_x, y=mean_r2, label=mean_label, color=model),
            size=3.4, fontface="bold", vjust=0.5, show.legend=FALSE) +
  scale_color_manual(values=c("Ordinary Least Squares"=GREEN,
                              "Random forest"=PINK)) +
  scale_x_discrete(expand=expansion(add=0.6)) +
  labs(title="Ordinary Least Squares and Random Forest Score Similarly Across Folds",
       x=NULL, y="Held Out R² (Dollar Scale)") +
  guides(color="none")

In [64]:
metrics_df=pd.DataFrame({
    "Model": ["Ordinary Least Squares","Random Forest","Neural Network"],
    "R²": [ols_log_test_r2, rf_log_test_r2, nn_test_r2],
    "Mean Absolute Error ($)": [ols_log_test_mae, rf_log_test_mae, nn_test_mae],
})

style_table(GT(metrics_df)
  .tab_header(title="Model Performance on the Held Out Counties")
  .fmt_number(columns="R²", decimals=3)
  .fmt_currency(columns="Mean Absolute Error ($)", decimals=0)
  .tab_source_note("Ordinary Least Squares and Random Forest are fit on the log target and scored on the dollar scale after back transformation.")
  .tab_source_note("Neural network results are from this single split only; Table 3 reports cross validated results for the other two models."))

This is the ladder’s stopping rule working as designed: a random forest can represent any interaction or curvature the data contain, yet it finds nothing beyond what the log transformation already captured. The binned comparison in <a href="#fig-binned" class="quarto-xref">Figure 19</a> confirms the same conclusion, this time in the units of the outcome itself. Specifically, the relationship is not monotonic across these bins: purchasing power rises with routine cognitive share through the lower part of its observed range, peaks near 20 percent, and then declines as the share continues to rise. The forest’s predictions track this same shape closely rather than discovering a different pattern of their own, which is further evidence that the flexible model adds little beyond what the log linear specification already captures.

In [65]:
binned_ml=pd.DataFrame({
    "rc": X_test["routine_cognitive_share"].values,
    "Actual": y_test.values,
    "Random Forest": y_pred_rf_log_dollars,
})
binned_ml["bin"]=pd.qcut(binned_ml["rc"], 10, labels=False)
binned_means=binned_ml.groupby("bin")[["rc","Actual","Random Forest"]].mean().reset_index(drop=True)
binned_long=binned_means.melt(id_vars="rc", value_vars=["Actual","Random Forest"],
                              var_name="series", value_name="pp")

In [66]:
%%R -i binned_long -i rf_log_test_r2 -i rf_log_test_mae -w 9 -h 4.5 -u in -r 150
OBSERVED <- "#252525"
binned_long$rc_pct <- binned_long$rc*100
label_pts <- binned_long[binned_long$rc==max(binned_long$rc),]
rf_stat_label <- sprintf("Random Forest: R² = %.3f, MAE = $%s",
                          rf_log_test_r2, formatC(round(rf_log_test_mae), format="d", big.mark=","))

ggplot(binned_long, aes(x=rc_pct, y=pp, color=series)) +
  annotate("text", x=max(binned_long$rc_pct), y=max(binned_long$pp)*0.975,
           label=rf_stat_label, hjust=1, size=3.4, color=PINK, fontface="bold") +
  geom_line(aes(linewidth=series)) +
  geom_point(aes(size=series)) +
  geom_text(data=label_pts, aes(label=series), hjust=-0.15, size=3.4, fontface="bold", show.legend=FALSE) +
  scale_color_manual(values=c("Actual"=OBSERVED, "Random Forest"=PINK), guide="none") +
  scale_linewidth_manual(values=c("Actual"=1.0, "Random Forest"=0.7), guide="none") +
  scale_size_manual(values=c("Actual"=1.8, "Random Forest"=1.4), guide="none") +
  scale_x_continuous(labels=function(v) sprintf("%d%%", round(v)),
                      breaks=scales::pretty_breaks(n=8),
                      expand=expansion(mult=c(0.02, 0.28))) +
  scale_y_continuous(labels=function(v) sprintf("$%s", formatC(round(v), format="d", big.mark=","))) +
  labs(title="Actual vs. Random Forest Purchasing Power by Routine Cognitive Share",
       x="Routine Cognitive Share (Percent)", y="Purchasing Power")

## What carries the signal

Two measurements say how much the task groups contribute. The first removes them and refits: <a href="#fig-ablation" class="quarto-xref">Figure 20</a> shows what the random forest loses in held out R² when each block of features is taken away. Removing poverty and unemployment costs 0.178, more than four times the 0.042 lost by removing the task groups. The economic controls therefore carry more predictive information, but the task groups still contribute beyond them, answering the question the objection above raised.

In [67]:
ablation_df=pd.DataFrame({
    "model": ["Task groups","Poverty and unemployment"],
    "r2": [0.834, 0.698],
})
ablation_df["full_r2"]=0.876
ablation_df["drop"]=ablation_df["full_r2"]-ablation_df["r2"]

In [68]:
%%R -i ablation_df -w 7 -h 3 -u in -r 150
ablation_df$model <- factor(ablation_df$model, levels=c("Task groups","Poverty and unemployment"))

ggplot(ablation_df, aes(x=drop, y=model, fill=model)) +
  geom_col(width=0.55) +
  geom_text(aes(label=sprintf("%.3f", drop)), hjust=-0.25, size=4.0, fontface="bold", color="#252525") +
  scale_fill_manual(values=c("Task groups"=GREEN, "Poverty and unemployment"=ORANGE), guide="none") +
  scale_x_continuous(limits=c(0, 0.20), breaks=seq(0, 0.20, 0.05), expand=c(0,0)) +
  labs(title="Loss in Held Out R² When Each Feature Block Is Removed",
       x="Loss in Held Out R²", y=NULL)

The second measurement leaves the model intact and destroys the information instead. <a href="#fig-importance" class="quarto-xref">Figure 21</a> ranks what the random forest relies on, using the grouped permutation approach from <a href="#sec-analysis" class="quarto-xref">Section 4</a> so that indicator blocks and the task groups are each scored as a unit. These two interpretability figures are computed on the raw target forest rather than the log target one, since the two score within 0.001 of each other on held out counties and the raw target version reports directly in dollars. In that ranking, poverty rate carries the largest single drop, consistent with its dominance in the regression. The modeled task group proportions together come next at 0.153, ahead of the year block at 0.128, and state adds almost nothing at 0.008 once everything else is present.

The two measurements disagree in magnitude, and the reason is instructive. Permuting the task groups costs 0.153, while removing them entirely and refitting costs only 0.042. The gap exists because a refitted model can lean harder on poverty, unemployment, and time to recover much of what the task groups were carrying. The permutation figure therefore measures how much the fitted model uses the task groups, and the ablation measures how much of that is unique to them. Both measurements agree that the task groups function together as a set, which is what parts of a whole should do.

In [69]:
pov_idx=list(X_test.columns).index("poverty_rate")
unemp_idx=list(X_test.columns).index("unemployment_rate")

importance_df=pd.DataFrame({
    "feature": ["Poverty Rate","Task Groups (Joint)","Year (Joint)",
                "Unemployment Rate","State (Joint)"],
    "block": ["Economic controls","Task groups","Structural indicators",
              "Economic controls","Structural indicators"],
    "drop": [perm_imp.importances_mean[pov_idx], joint_drop, year_drop,
             perm_imp.importances_mean[unemp_idx], state_drop],
}).sort_values("drop")

In [70]:
%%R -i importance_df -w 8 -h 3.5 -u in -r 150
importance_df$feature <- factor(importance_df$feature, levels=importance_df$feature)
importance_df$block <- factor(importance_df$block,
    levels=c("Task groups","Economic controls","Structural indicators"))

ggplot(importance_df, aes(x=drop, y=feature, fill=block)) +
  geom_col(width=0.6) +
  geom_text(aes(label=sprintf("%.3f", drop)), hjust=-0.25, size=3.4, fontface="bold", color="#252525") +
  scale_fill_manual(values=c("Task groups"=GREEN, "Economic controls"=ORANGE,
                             "Structural indicators"=GREY)) +
  scale_x_continuous(expand=expansion(mult=c(0, 0.15))) +
  labs(title="Feature Importance by Grouped Permutation",
       x="Drop in Held Out R² When Permuted", y=NULL, fill=NULL) +
  theme(legend.position="bottom")

<a href="#fig-pdp" class="quarto-xref">Figure 22</a> completes the picture with the forest’s partial dependence on the three non-reference task groups. Each curve shows predicted purchasing power as one group’s value moves across its observed range with all other features held at their values. Across that range, the curves decline smoothly and near monotonically, with no thresholds or reversals. That shape is what a proportional association implies, and it is why the flexible model could not beat the log linear one. Because the four groups are parts of a whole, moving one while holding the others fixed implies a compensating change in the omitted reference group. These curves therefore describe the model’s behavior rather than a combination any county could occupy.

In [71]:
from sklearn.inspection import partial_dependence

pdp_frames=[]
for term, name in zip(["routine_cognitive_share","routine_manual_share","non_routine_manual_share"],
                      ["Routine Cognitive","Routine Manual","Non-Routine Manual"]):
    pd_res=partial_dependence(best_rf, X_test, [term], kind="average", grid_resolution=40)
    grid=pd_res["grid_values"][0] if "grid_values" in pd_res else pd_res["values"][0]
    pdp_frames.append(pd.DataFrame({"x":grid,
                                    "y":pd_res["average"][0],
                                    "group":name}))
pdp_df=pd.concat(pdp_frames, ignore_index=True)

In [72]:
%%R -i pdp_df -w 10 -h 3.5 -u in -r 150
pdp_df$group <- factor(pdp_df$group, levels=c("Routine Cognitive","Routine Manual","Non-Routine Manual"))
PDP_COLORS <- TASK_COLORS[c("Routine Cognitive","Routine Manual","Non-Routine Manual")]

ggplot(pdp_df, aes(x=x, y=y, color=group)) +
  geom_line(linewidth=0.8, show.legend=FALSE) +
  facet_wrap(~group, scales="free_x") +
  scale_color_manual(values=PDP_COLORS) +
  scale_y_continuous(labels=function(v) sprintf("$%s", formatC(round(v), format="d", big.mark=","))) +
  labs(title="Partial Dependence on Each Task Group",
       x="Task Group Value", y="Predicted Purchasing Power")

## The differences are durable

The final question is whether these associations describe a stable feature of places or a moment in time. Three pieces of evidence say stable. <a href="#fig-taskarea" class="quarto-xref">Figure 23</a> shows the panel average of each task group by year, which moves little across fifteen years apart from the 2009 to 2010 step produced by the Census occupation coding change described in <a href="#sec-data" class="quarto-xref">Section 3</a>.

In [73]:
area_df=(df.groupby("year")[["routine_cognitive_share","routine_manual_share",
                             "non_routine_cognitive_share","non_routine_manual_share"]]
           .mean().reset_index()
           .melt(id_vars="year", var_name="group", value_name="share"))
area_names={"routine_cognitive_share":"Routine Cognitive",
            "routine_manual_share":"Routine Manual",
            "non_routine_cognitive_share":"Non-Routine Cognitive",
            "non_routine_manual_share":"Non-Routine Manual"}
area_df["group"]=area_df["group"].map(area_names)
area_df["period"]=np.where(area_df["year"]<=2019, "pre", "post")

# narrow, deliberate gap right at the missing 2020 year, rather than the
# full two year blank span a plain pre/2021 split would otherwise leave
gap_rows=[]
for grp in area_df["group"].unique():
    before=area_df.loc[(area_df["group"]==grp) & (area_df["year"]==2019), "share"].iloc[0]
    after=area_df.loc[(area_df["group"]==grp) & (area_df["year"]==2021), "share"].iloc[0]
    gap_rows.append({"year":2019.9, "group":grp, "share":before+(after-before)*0.45, "period":"pre"})
    gap_rows.append({"year":2020.1, "group":grp, "share":before+(after-before)*0.55, "period":"post"})
area_df=pd.concat([area_df, pd.DataFrame(gap_rows)], ignore_index=True).sort_values("year").reset_index(drop=True)

In [74]:
%%R -i area_df -w 9.5 -h 4.7 -u in -r 150
TASKAREA_COLORS <- TASK_COLORS
group_levels <- c("Non-Routine Cognitive","Non-Routine Manual","Routine Cognitive","Routine Manual")
area_df$group <- factor(area_df$group, levels=group_levels)

pre_df  <- area_df[area_df$period=="pre",]
post_df <- area_df[area_df$period=="post",]

label_year <- 2016
label_df <- area_df[area_df$year==label_year,]
label_df <- label_df[match(group_levels, label_df$group),]
label_df$cum <- cumsum(label_df$share)
label_df$mid <- label_df$cum - label_df$share/2
# Non-Routine Cognitive and Routine Cognitive are the light members of their hue
# families, so white labels lose contrast there; the two manual groups stay dark.
label_df$text_color <- ifelse(label_df$group %in% c("Non-Routine Manual","Routine Manual"),
                               "white", "#0B0B0B")

ggplot(area_df, aes(x=year, y=share, fill=group)) +
  geom_area(data=pre_df, stat="identity", position=position_stack(reverse=TRUE), linewidth=0) +
  geom_area(data=post_df, stat="identity", position=position_stack(reverse=TRUE), linewidth=0) +
  geom_text(data=label_df, aes(x=label_year, y=mid, label=group, color=text_color),
            hjust=0.5, size=3.3, fontface="bold") +
  scale_color_identity() +
  geom_vline(xintercept=2009.5, linetype="dashed", linewidth=1.0, color="#0B0B0B") +
  geom_vline(xintercept=2020, linetype="dashed", linewidth=0.6, color="grey45") +
  annotate("text", x=2009.5, y=1.06, label="Census coding change", hjust=0.5, vjust=0.5, size=3.1, fontface="bold", color="#0B0B0B") +
  annotate("text", x=2020, y=1.06, label="No data", hjust=0.5, vjust=0.5, size=3.1, fontface="bold", color="grey45") +
  scale_fill_manual(values=TASKAREA_COLORS, guide="none") +
  scale_y_continuous(labels=function(v) sprintf("%.0f%%", v*100), breaks=seq(0, 1, 0.25),
                      limits=c(0, 1.13), expand=c(0,0)) +
  scale_x_continuous(breaks=2008:2023, limits=c(2008, 2023.3), expand=c(0,0)) +
  labs(title="Average County Task Group Value by Year",
       x="Year", y="Mean Group Value Across Counties") +
  theme(axis.text.x=element_text(angle=0, hjust=0.5),
        plot.margin=margin(10, 5.5, 5.5, 5.5))

Individual counties hold their positions as well, not just the national aggregate. <a href="#tbl-rankcorr" class="quarto-xref">Table 6</a> reports the rank correlation of each group’s county ordering between 2010 and 2023, computed from 2010 onward to step over the coding change. Routine manual and non-routine cognitive hold their orderings most strongly, so the counties with the most of each in 2010 are largely the same counties in 2023. Non-routine manual is more mobile, and routine cognitive reorders the most, which is consistent with it being the one group carrying substantial within county movement in the variance decomposition of <a href="#sec-analysis" class="quarto-xref">Section 4</a>.

In [75]:
from scipy.stats import spearmanr

wide_2010=df[df["year"]==2010].set_index("county_fips")
wide_2023=df[df["year"]==2023].set_index("county_fips")
common=wide_2010.index.intersection(wide_2023.index)

rank_rows=[]
for col, name in area_names.items():
    rho=spearmanr(wide_2010.loc[common, col], wide_2023.loc[common, col]).statistic
    rank_rows.append({"Task group":name, "Rank correlation, 2010 to 2023":rho})
rankcorr_df=pd.DataFrame(rank_rows)

style_table(GT(rankcorr_df)
  .tab_header(title="County Ordering Stability, 2010 to 2023")
  .fmt_number(columns="Rank correlation, 2010 to 2023", decimals=2)
  .cols_align(align="right", columns="Rank correlation, 2010 to 2023")
  .tab_source_note("Computed from 2010 onward to avoid the Census occupation coding change."))

<a href="#fig-taskchange-map" class="quarto-xref">Figure 24</a> shows this stability geographically, using the same two years as <a href="#tbl-rankcorr" class="quarto-xref">Table 6</a>. Routine manual and non-routine manual, the two groups with almost no net movement in <a href="#fig-taskarea" class="quarto-xref">Figure 23</a>, show small changes that run in both directions across counties, consistent with variation that sits between counties rather than within them. Routine cognitive and non-routine cognitive, the pair whose national levels moved most in <a href="#fig-taskarea" class="quarto-xref">Figure 23</a>, show a change that is both larger and far more uniform in direction across the map, matching the within county movement identified in the variance decomposition of <a href="#sec-analysis" class="quarto-xref">Section 4</a>.

In [76]:
import geopandas as gpd

change_wide=(wide_2023.loc[common, list(area_names.keys())]
             -wide_2010.loc[common, list(area_names.keys())])*100
change_wide.columns=[area_names[c] for c in change_wide.columns]
change_long=change_wide.reset_index().melt(id_vars="county_fips", var_name="group", value_name="change")
change_long["fips"]=change_long["county_fips"].astype(str).str.zfill(5)

gdf_change=gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip")
gdf_change=gdf_change.rename(columns={"GEOID":"fips"})
gdf_change=gdf_change[~gdf_change["STATEFP"].isin(["02","15","60","66","69","72","78"])]

# cross every continental county with all four groups so counties outside the panel
# still render, grey, in every facet, matching the coverage convention in fig-afford-map
groups_order=list(area_names.values())
county_group_grid=gdf_change[["fips","geometry"]].merge(
    pd.DataFrame({"group":groups_order}), how="cross")
merged_change=county_group_grid.merge(change_long[["fips","group","change"]],
                                      on=["fips","group"], how="left")
os.makedirs("output", exist_ok=True)
merged_change.to_file("output/taskchange_map.geojson", driver="GeoJSON")

change_bound=float(change_long["change"].abs().quantile(0.98))

In [77]:
%%R -i change_bound -w 9.5 -h 7 -u in -r 150
suppressMessages(library(sf))

change_sf <- st_read("output/taskchange_map.geojson", quiet=TRUE)
change_sf$group <- factor(change_sf$group,
    levels=c("Routine Cognitive","Routine Manual","Non-Routine Cognitive","Non-Routine Manual"))

ggplot(change_sf) +
  geom_sf(aes(fill=change), color="white", linewidth=0.03) +
  facet_wrap(~group, ncol=2) +
  scale_fill_distiller(palette="PuOr", direction=1, limits=c(-change_bound, change_bound),
                        oob=scales::squish, na.value="#eeeeee",
                        labels=function(v) sprintf("%+.0fpp", v),
                        name="Change in Task Group Value (Percentage Points)") +
  coord_sf(crs=st_crs(5070), datum=NA) +
  theme_void() +
  labs(title="Change in Task Group Value by County, 2010 to 2023") +
  theme(legend.position="bottom", legend.key.width=unit(1.3,"cm"),
        plot.title=element_text(face="bold", size=13, hjust=0.5),
        strip.text=element_text(size=11, face="bold"),
        plot.background=element_rect(fill="#FCFCFB", color=NA))

Finally, the coefficients themselves hold across time. <a href="#fig-stability" class="quarto-xref">Figure 25</a> refits the log panel regression on the pre pandemic window, 2010 to 2019, and the post pandemic window, 2021 to 2023, the same windows named in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. Reporting these in percent rather than dollars keeps the three windows comparable, since nominal purchasing power rose substantially across the panel and a dollar coefficient in the later window is measured against a larger base. The task group ordering is identical in both windows and in the full panel, and the magnitudes move modestly, so the association is not an artifact of any one period, recession, recovery, or pandemic era.

In [78]:
def fit_window_log(frame):
    Xw=pd.concat([frame[res_cols],
                  pd.get_dummies(frame["year"], prefix="year", drop_first=True).astype(float)], axis=1)
    Xw=sm.add_constant(Xw)
    return sm.OLS(np.log(frame["affordability_salary"]), Xw).fit(
        cov_type="cluster", cov_kwds={"groups": frame["county_fips"]})

windows={"Full panel": reg,
         "2010 to 2019": reg[(reg["year"]>=2010)&(reg["year"]<=2019)],
         "2021 to 2023": reg[reg["year"]>=2021]}

stab_rows=[]
for term, name in zip(group_terms, group_names):
    row={"Task group":name}
    for wname, frame in windows.items():
        b=fit_window_log(frame).params[term]
        row[wname]=(np.exp(b*0.01)-1)*100
    stab_rows.append(row)
stability_df=pd.DataFrame(stab_rows)

stab_long=stability_df.melt(id_vars="Task group", var_name="window", value_name="value")
range_df=stability_df.assign(
    lo=stability_df[["Full panel","2010 to 2019","2021 to 2023"]].min(axis=1),
    hi=stability_df[["Full panel","2010 to 2019","2021 to 2023"]].max(axis=1),
)[["Task group","lo","hi"]]

In [79]:
%%R -i stab_long -i range_df -w 8 -h 3.6 -u in -r 150
BLACK <- "#252525"

group_order <- c("Routine Manual", "Routine Cognitive", "Non-Routine Manual")
stab_long$window <- factor(stab_long$window, levels=c("2010 to 2019", "Full panel", "2021 to 2023"))
stab_long$`Task group` <- factor(stab_long$`Task group`, levels=group_order)
range_df$`Task group` <- factor(range_df$`Task group`, levels=group_order)

window_colors <- c("2010 to 2019"=BLUE, "Full panel"=BLACK, "2021 to 2023"=ORANGE)
window_shapes <- c("2010 to 2019"=16, "Full panel"=18, "2021 to 2023"=16)
window_sizes  <- c("2010 to 2019"=3.2, "Full panel"=4.6, "2021 to 2023"=3.2)

ggplot() +
  geom_segment(data=range_df, aes(x=lo, xend=hi, y=`Task group`, yend=`Task group`),
               color="#C8C8C5", linewidth=1.1) +
  geom_point(data=stab_long, aes(x=value, y=`Task group`, color=window, shape=window, size=window)) +
  scale_color_manual(values=window_colors, name=NULL) +
  scale_shape_manual(values=window_shapes, name=NULL) +
  scale_size_manual(values=window_sizes, guide="none") +
  scale_x_continuous(labels=function(v) sprintf("%.1f%%", v)) +
  labs(title="Task Group Coefficients Across Time Windows",
       x="Percent Change per Point Shifted Out of Non-Routine Cognitive Work", y=NULL) +
  theme(legend.position="bottom")

Taken together, the results show that this pattern is not limited to a single year or model. Counties with more routine and manual work tended to have lower purchasing power at the beginning of the study period, and that pattern remained largely unchanged fifteen years later. <a href="#sec-conclusions" class="quarto-xref">Section 6</a> discusses what these findings mean, their limits, and what questions remain.

# Conclusions

## Summary of findings

The introduction opened with national concern about automation. At the county level, the evidence shows that a county’s task groups are associated with real differences in residents’ purchasing power. Across 848 counties and fifteen years, counties with higher concentrations of routine and manual work tend to have lower purchasing power than counties where non-routine cognitive work dominates. This pattern persists after accounting for poverty, unemployment, population, and year. At the panel mean, a one percentage point shift from non-routine cognitive to non-routine manual work is associated with roughly \$846 less in purchasing power. The diagnostics suggest this relationship is better represented as proportional rather than linear in dollars. Neither the random forest nor the neural network improved on the log linear specification, so the added complexity provided little benefit.

## Contributions

The introduction identified three gaps between the existing task based literature and a county level analysis of purchasing power. This study addresses each of them. First, prior work has typically evaluated labor market outcomes using nominal wages, employment, or related measures that do not account for geographic differences in prices. Here, the outcome is median household income adjusted by the BEA Regional Price Parities, allowing the analysis to compare what income can purchase across counties rather than income alone. This distinction matters because two counties with similar nominal incomes may provide very different standards of living once local prices are considered.

Second, much of the task based literature has relied on broader labor market units such as commuting zones. This study instead uses the county as the unit of analysis. That finer geographic scale makes it possible to distinguish differences within the same metropolitan region and to connect task groups more directly to the economic conditions experienced by local residents. The county year panel also allows those differences to be followed from 2008 through 2023 rather than treated as a single cross sectional comparison.

Third, the study does not assume that the relationship between task groups and purchasing power is adequately represented by a simple linear model. The functional form is evaluated directly through diagnostic testing, a logged specification, and comparison with more flexible machine learning models. The results favor a proportional interpretation of the association, while the random forest and neural network provide little improvement once that form is accounted for. This strengthens the case for retaining the more interpretable log linear model rather than adding complexity without a corresponding gain in performance. These extensions move the task framework from a description of occupational structure toward a county level measure. That measure can be related directly to local purchasing power using publicly available and reproducible data.

The findings also have practical relevance. For policymakers, task groups provide information about local economic conditions that is not fully captured by poverty or unemployment alone. Counties with similar levels of conventional economic distress can still differ in the kinds of work their residents perform and in the purchasing power associated with that employment structure. The task measures therefore offer an additional way to identify places where economic vulnerability may not be fully visible in standard indicators.

For economic development agencies and other organizations making place based investment decisions, the purchasing power measure adds a second distinction. Nominal income can make some counties appear stronger or weaker than they are once local prices are taken into account. Combining purchasing power with task groups helps identify where those differences occur and what features of the local labor market are associated with them. The broader contribution of the study is an additional lens on existing measures of economic distress. Task groups capture a relatively persistent feature of local economies, one that helps explain differences in purchasing power beyond poverty, unemployment, and population alone.

## Limitations

Five limitations bound what the results can support. The findings are associations, not causal effects, since counties were not assigned their task groups. Any factor tied to both the work a county contains and its purchasing power could account for part of the relationship. The panel covers only counties above 65,000 residents, so rural counties are underrepresented and the estimates describe the more populous three quarters of the population rather than counties in general. The panel regression carries no state effects, so its estimates absorb whatever varies systematically by state. The predictive models, which do include state indicators, suggest that variation is modest once poverty, unemployment, and time are present, but the two specifications are not identical on this point. Purchasing power is also not deflated to a constant base year, so the year indicators absorb national price drift alongside every other change common to a year, rather than the panel correcting for inflation directly. <a href="#sec-analysis" class="quarto-xref">Section 4</a> treats them as nuisance parameters for this reason. That choice means the year effects cannot be read as a measure of purchasing power’s growth over time. And a majority of panel rows carry a state level price parity rather than a local one, so the outcome is measured more coarsely in nonmetropolitan counties, and the analysis does not separate the two.

## Ethical considerations

The design also carries ethical boundaries that shape how the results should be read. The unit of analysis is the county, so every claim describes places rather than people, and inferring anything about an individual worker from these results would be an ecological fallacy. The population threshold means the smallest and most rural counties are absent, and any application of these findings to such places would be extrapolation beyond the data. The associational framing is not a hedge but a constraint, since presenting these estimates as causal could misdirect policy toward changing a county’s task groups when the underlying driver may lie elsewhere. And the study uses public aggregate data only, so no individual’s information enters the analysis at any point.

## Future directions

Two directions follow from the limitations. The coverage gap could be narrowed with satellite imagery. Jean et al. ([2016](#ref-Jean2016)) estimate local economic conditions from daytime and nighttime imagery where survey data are thin. Applied here, that approach could produce purchasing power estimates for the counties below the American Community Survey threshold, which would show whether the geography reported above extends to the places this panel cannot see. It would not extend the association itself, since occupational estimates remain unavailable for those counties. But confirming that the outcome pattern continues below the threshold would tell us the geography is a feature of the country rather than of the sample. Establishing causation is the harder question. It would require a source of variation in task groups that is unrelated to purchasing power, such as plant openings and closures or technology adoption shocks. That would call for a separate study, using different data and a design capable of isolating plausible exogenous changes in local task groups.

## Conclusion

This study establishes that task groups are a durable, measurable characteristic of counties, and that they carry real information about what residents can afford. That relationship is an association, not a causal claim, since counties differ in far more than their task groups. The counties whose work was most routine and most manual entered the study period with less purchasing power, and fifteen years later they largely still do. Because a county’s task groups change so little over time, the purchasing power gap they mark does not close on its own; it behaves as a structural feature of a place rather than a passing downturn. Anyone reading only the poverty rate is looking at an incomplete picture of where purchasing power in America is strained, and a county’s task groups are part of what completes it.

# References

Acemoglu, Daron, and David Autor. 2011. “Skills, Tasks and Technologies: Implications for Employment and Earnings.” In *Handbook of Labor Economics*, edited by David Card and Orley Ashenfelter, 4:1043–1171. Elsevier. <https://doi.org/10.1016/S0169-7218(11)02410-5>.

Autor, David H., and David Dorn. 2013. “The Growth of Low-Skill Service Jobs and the Polarization of the US Labor Market.” *American Economic Review* 103 (5): 1553–97. <https://doi.org/10.1257/aer.103.5.1553>.

Autor, David H., Frank Levy, and Richard J. Murnane. 2003. “The Skill Content of Recent Technological Change: An Empirical Exploration.” *The Quarterly Journal of Economics* 118 (4): 1279–1333. <https://doi.org/10.1162/003355303322552801>.

Jean, Neal, Marshall Burke, Michael Xie, W. Matthew Davis, David B. Lobell, and Stefano Ermon. 2016. “Combining Satellite Imagery and Machine Learning to Predict Poverty.” *Science* 353 (6301): 790–94. <https://doi.org/10.1126/science.aaf7894>.

National Center for O\*NET Development, U.S. Department of Labor. 2024. “O\*NET Database.” <https://www.onetcenter.org/database.html>.

Saad, Lydia. 2023. “More U.S. Workers Fear Technology Making Their Jobs Obsolete.” Gallup. <https://news.gallup.com/poll/510551/workers-fear-technology-making-jobs-obsolete.aspx>.

Smith, Aaron, and Monica Anderson. 2017. “Automation in Everyday Life.” Pew Research Center. <https://www.pewresearch.org/internet/2017/10/04/automation-in-everyday-life/>.

U.S. Bureau of Economic Analysis. 2024. “Regional Price Parities by State and Metro Area.” <https://www.bea.gov/data/prices-inflation/regional-price-parities-state-and-metro-area>.

U.S. Bureau of Labor Statistics. 2024. “Local Area Unemployment Statistics.” <https://www.bls.gov/lau/>.

U.S. Census Bureau. 2023a. “Core Based Statistical Area Delineation Files.” <https://www.census.gov/geographies/reference-files/time-series/demo/metro-micro/delineation-files.html>.

———. 2023b. “Vintage 2023 Population Estimates.” <https://www.census.gov/programs-surveys/popest.html>.

———. 2024. “American Community Survey 1-Year Estimates.” <https://www.census.gov/programs-surveys/acs>.